P4-0 — Global setup, imports, random seed

In [1]:
# ============================================================
# P4-0 — Global setup, imports, random seed
# Automatic Model Training Pipeline
# ============================================================

import os
import re
import json
import math
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("P4-0 ready")
print("Python environment ready")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

P4-0 ready
Python environment ready
Pandas: 3.0.3
NumPy: 2.3.5


P4-1 — Define AutoTrainingConfig

In [2]:
# ============================================================
# P4-1 — Define AutoTrainingConfig
# ============================================================

@dataclass
class AutoTrainingConfig:
    # ---- Identity ----
    pipeline_name: str = "automatic_model_training_pipeline"
    process_id: str = "apb"

    # ---- Input ----
    data_path: str = "apb_data_cleaned.xlsx"

    # ---- Required raw schema ----
    occurred_col: str = "occurred"
    machine_col: str = "mc_no"
    status_col: str = "mc_status"
    process_col: str = "process"

    # ---- Status definition ----
    # 😮 Normal statuses are excluded from prediction targets.
    normal_statuses: list = field(default_factory=lambda: ["Alarm Other"])

    # "auto" means all statuses not in normal_statuses become target statuses.
    # Or set explicit list, e.g. ["mc_alarm", "mc_stop"]
    target_statuses: str | list = "auto"

    # 😮 Optional status aliases.
    # Keys and values will be normalized automatically.
    status_alias_map: dict = field(default_factory=lambda: {
        "Alarm Coating": "alarm_coating",
        "Alarm Palette": "alarm_palette",
        "Alarm Other": "alarm_other",
        
    })

    # 😮 Dashboard/display only. Does not affect training target.
    hidden_statuses: list = field(default_factory=lambda: ["Alarm Other"])

    # ---- Data validation ----
    allow_multiple_processes: bool = False
    min_rows: int = 1_000
    min_machines: int = 1
    min_target_rows: int = 100

    # ---- Deduplication ----
    # exact_duplicate_rows: same process, mc_no, occurred, mc_status
    # same_timestamp_conflict_policy:
    #   "drop_conflicts" = drop rows where same process+mc_no+occurred has multiple statuses
    #   "keep"           = keep them, later target labeling may remove zero/invalid gaps
    same_timestamp_conflict_policy: str = "drop_conflicts"

    # ---- Later batches will use these ----
    rolling_windows_min: list = field(default_factory=lambda: [5, 15, 30, 60, 120])
    train_frac: float = 0.70
    val_frac: float = 0.15
    test_frac: float = 0.15

    target_clip_quantile: float = 0.99

    confidence_threshold: float = 0.60

    artifact_dir: str = "./artifacts_phase4"
    artifact_schema_version: str = "phase4_auto_v1"
    feature_version: str = "auto_features_v1"


CFG = AutoTrainingConfig()

print("P4-1 ready")
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))

P4-1 ready
{
  "pipeline_name": "automatic_model_training_pipeline",
  "process_id": "apb",
  "data_path": "apb_data_cleaned.xlsx",
  "occurred_col": "occurred",
  "machine_col": "mc_no",
  "status_col": "mc_status",
  "process_col": "process",
  "normal_statuses": [
    "Alarm Other"
  ],
  "target_statuses": "auto",
  "status_alias_map": {
    "Alarm Coating": "alarm_coating",
    "Alarm Palette": "alarm_palette",
    "Alarm Other": "alarm_other"
  },
  "hidden_statuses": [
    "Alarm Other"
  ],
  "allow_multiple_processes": false,
  "min_rows": 1000,
  "min_machines": 1,
  "min_target_rows": 100,
  "same_timestamp_conflict_policy": "drop_conflicts",
  "rolling_windows_min": [
    5,
    15,
    30,
    60,
    120
  ],
  "train_frac": 0.7,
  "val_frac": 0.15,
  "test_frac": 0.15,
  "target_clip_quantile": 0.99,
  "confidence_threshold": 0.6,
  "artifact_dir": "./artifacts_phase4",
  "artifact_schema_version": "phase4_auto_v1",
  "feature_version": "auto_features_v1"
}


P4-2 — Load cleaned dataset

In [3]:
# ============================================================
# P4-2 — Load cleaned dataset
# ============================================================

def load_table(path: str) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    elif suffix in [".parquet", ".pq"]:
        return pd.read_parquet(path)
    elif suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    else:
        raise ValueError(
            f"Unsupported file type: {suffix}. "
            "Supported: .csv, .parquet, .pq, .xlsx, .xls"
        )


df_raw = load_table(CFG.data_path)

print("P4-2 ready")
print("Loaded:", CFG.data_path)
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))

display(df_raw.head())
display(df_raw.dtypes.to_frame("dtype"))

P4-2 ready
Loaded: apb_data_cleaned.xlsx
Shape: (154, 4)
Columns: ['occurred', 'mc_no', 'mc_status', 'process']


,occurred,mc_no,mc_status,process
0,2026-04-03 02:13:00,BLDC-CT-02,Alarm Palette,apb_test
1,2026-04-03 00:05:00,BLDC-CT-03,Alarm Coating,apb_test
2,2026-04-04 04:10:00,BLDC-CT-02,Alarm Other,apb_test
3,2026-04-04 00:10:00,BLDC-CT-03,Alarm Coating,apb_test
4,2026-04-03 23:10:00,BLDC-CT-02,Alarm Coating,apb_test


,dtype
occurred,datetime64[us]
mc_no,str
mc_status,str
process,str


P4-3 — Standardize schema and normalize status names

In [4]:
# ============================================================
# P4-3 — Standardize schema and normalize status names
# ============================================================

def normalize_col_name(x: str) -> str:
    return str(x).strip()


def normalize_status_text(x) -> str:
    """
    Canonical status text for modeling.
    Examples:
      'RUN'         -> 'run'
      'mc_fullWork' -> 'mc_fullwork'
      'm/c stop'   -> 'm_c_stop'
      'mc waitpart'-> 'mc_waitpart'
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", "_", s)
    s = s.replace("/", "_")
    s = s.replace("-", "_")
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")
    return s


def sanitize_feature_token(x) -> str:
    """
    Stable feature-name-safe token.
    Used later for dynamic one-hot columns.
    """
    s = normalize_status_text(x)
    if pd.isna(s):
        return "missing"
    s = re.sub(r"[^a-zA-Z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_")


def canonicalize_status(x, alias_map: dict) -> str:
    s = normalize_status_text(x)

    alias_norm = {
        normalize_status_text(k): normalize_status_text(v)
        for k, v in alias_map.items()
    }

    return alias_norm.get(s, s)


def standardize_schema(df: pd.DataFrame, cfg: AutoTrainingConfig) -> pd.DataFrame:
    out = df.copy()

    # Strip column names
    out.columns = [normalize_col_name(c) for c in out.columns]

    required_raw_cols = [
        cfg.occurred_col,
        cfg.machine_col,
        cfg.status_col,
        cfg.process_col,
    ]

    missing = [c for c in required_raw_cols if c not in out.columns]
    if missing:
        raise ValueError(
            f"Missing required raw columns: {missing}. "
            f"Available columns: {list(out.columns)}"
        )

    # Rename to standard internal schema
    rename_map = {
        cfg.occurred_col: "occurred",
        cfg.machine_col: "mc_no",
        cfg.status_col: "mc_status_raw",
        cfg.process_col: "process",
    }

    out = out.rename(columns=rename_map)

    # Keep only standard required columns for this pipeline phase
    out = out[["occurred", "mc_no", "mc_status_raw", "process"]].copy()

    # Parse timestamp
    out["occurred"] = pd.to_datetime(out["occurred"], errors="coerce")

    # Normalize text columns
    out["mc_no"] = out["mc_no"].astype(str).str.strip()
    out["process"] = out["process"].astype(str).str.strip().str.lower()
    out["mc_status_raw"] = out["mc_status_raw"].astype(str).str.strip()

    # Canonical model status
    out["mc_status"] = out["mc_status_raw"].apply(
        lambda x: canonicalize_status(x, cfg.status_alias_map)
    )

    # Stable feature token for future feature engineering
    out["mc_status_token"] = out["mc_status"].apply(sanitize_feature_token)

    return out


df_std = standardize_schema(df_raw, CFG)

print("P4-3 ready")
print("Standardized shape:", df_std.shape)
print("Columns:", list(df_std.columns))

display(df_std.head())
display(df_std.dtypes.to_frame("dtype"))

print("\nRaw status values:")
display(df_std["mc_status_raw"].value_counts(dropna=False).to_frame("count"))

print("\nCanonical status values:")
display(df_std["mc_status"].value_counts(dropna=False).to_frame("count"))

P4-3 ready
Standardized shape: (154, 6)
Columns: ['occurred', 'mc_no', 'mc_status_raw', 'process', 'mc_status', 'mc_status_token']


,occurred,mc_no,mc_status_raw,process,mc_status,mc_status_token
0,2026-04-03 02:13:00,BLDC-CT-02,Alarm Palette,apb_test,alarm_palette,alarm_palette
1,2026-04-03 00:05:00,BLDC-CT-03,Alarm Coating,apb_test,alarm_coating,alarm_coating
2,2026-04-04 04:10:00,BLDC-CT-02,Alarm Other,apb_test,alarm_other,alarm_other
3,2026-04-04 00:10:00,BLDC-CT-03,Alarm Coating,apb_test,alarm_coating,alarm_coating
4,2026-04-03 23:10:00,BLDC-CT-02,Alarm Coating,apb_test,alarm_coating,alarm_coating


,dtype
occurred,datetime64[us]
mc_no,str
mc_status_raw,str
process,str
mc_status,str
mc_status_token,str



Raw status values:


,count
mc_status_raw,
Alarm Other,66
Alarm Coating,47
Alarm Palette,41



Canonical status values:


,count
mc_status,
alarm_other,66
alarm_coating,47
alarm_palette,41


P4-4 — Define normal_statuses and target_statuses

In [5]:
# ============================================================
# P4-4 — Define normal_statuses and target_statuses
# ============================================================

NORMAL_STATUSES = set(normalize_status_text(s) for s in CFG.normal_statuses)

ALL_STATUSES = sorted(df_std["mc_status"].dropna().unique().tolist())

if CFG.target_statuses == "auto":
    TARGET_STATUSES = set(s for s in ALL_STATUSES if s not in NORMAL_STATUSES)
else:
    TARGET_STATUSES = set(normalize_status_text(s) for s in CFG.target_statuses)

IGNORED_STATUSES = set(ALL_STATUSES) - NORMAL_STATUSES - TARGET_STATUSES

STATUS_ROLE = {}
for s in ALL_STATUSES:
    if s in NORMAL_STATUSES:
        STATUS_ROLE[s] = "normal"
    elif s in TARGET_STATUSES:
        STATUS_ROLE[s] = "target"
    else:
        STATUS_ROLE[s] = "ignored"

status_role_df = (
    pd.DataFrame({
        "mc_status": ALL_STATUSES,
        "role": [STATUS_ROLE[s] for s in ALL_STATUSES],
        "count": [int((df_std["mc_status"] == s).sum()) for s in ALL_STATUSES],
    })
    .sort_values(["role", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

print("P4-4 ready")
print("NORMAL_STATUSES:", sorted(NORMAL_STATUSES))
print("TARGET_STATUSES:", sorted(TARGET_STATUSES))
print("IGNORED_STATUSES:", sorted(IGNORED_STATUSES))

display(status_role_df)

P4-4 ready
NORMAL_STATUSES: ['alarm_other']
TARGET_STATUSES: ['alarm_coating', 'alarm_palette']
IGNORED_STATUSES: []


,mc_status,role,count
0,alarm_other,normal,66
1,alarm_coating,target,47
2,alarm_palette,target,41


P4-5 — Deduplicate exact duplicate rows

This removes exact duplicate event rows and conservatively drops same-machine/same-timestamp status conflicts without using any priority rule.

In [6]:
# ============================================================
# P4-5 — Deduplicate exact duplicate rows
# ============================================================

def deduplicate_events(df: pd.DataFrame, cfg: AutoTrainingConfig) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    # 1. Validate configuration upfront
    if cfg.same_timestamp_conflict_policy not in ["drop_conflicts", "keep"]:
        raise ValueError(
            "same_timestamp_conflict_policy must be either "
            "'drop_conflicts' or 'keep'"
        )

    out = df.copy()
    before_rows = len(out)

    exact_key = ["process", "mc_no", "occurred", "mc_status"]
    exact_duplicate_rows = int(out.duplicated(subset=exact_key, keep="first").sum())
    out = out.drop_duplicates(subset=exact_key, keep="first").copy()
    after_exact_rows = len(out)

    timestamp_key = ["process", "mc_no", "occurred"]
    conflict_groups = (
        out.groupby(timestamp_key)
        .agg(
            n_rows=("mc_status", "size"),
            n_status=("mc_status", "nunique"),
            statuses=("mc_status", lambda x: sorted(set(x))),
        )
        .reset_index()
    )
    conflict_groups = conflict_groups[conflict_groups["n_status"] > 1].copy()
    conflict_group_count = len(conflict_groups)

    # 2. Simplified conflict removal logic
    conflict_rows_removed = 0
    if cfg.same_timestamp_conflict_policy == "drop_conflicts" and conflict_group_count > 0:
        conflict_keys = conflict_groups[timestamp_key].copy()
        conflict_keys["_conflict"] = 1

        out = out.merge(conflict_keys, on=timestamp_key, how="left")
        conflict_rows_removed = int(out["_conflict"].eq(1).sum())
        out = out[out["_conflict"].isna()].drop(columns=["_conflict"]).copy()

    after_rows = len(out)
    out = (
        out.sort_values(["process", "mc_no", "occurred", "mc_status"])
        .reset_index(drop=True)
    )

    report = {
        "before_rows": before_rows,
        "exact_duplicate_rows_removed": exact_duplicate_rows,
        "after_exact_dedup_rows": after_exact_rows,
        "same_timestamp_conflict_groups": int(conflict_group_count),
        "same_timestamp_conflict_rows_removed": int(conflict_rows_removed),
        "after_dedup_rows": after_rows,
        "total_rows_removed": before_rows - after_rows,
        "same_timestamp_conflict_policy": cfg.same_timestamp_conflict_policy,
    }

    return out, report, conflict_groups


df_clean, dedup_report, timestamp_conflict_groups = deduplicate_events(df_std, CFG)

print("P4-5 ready")
print(json.dumps(dedup_report, indent=2, ensure_ascii=False))

if len(timestamp_conflict_groups) > 0:
    print("\nExample same-timestamp conflict groups:")
    display(timestamp_conflict_groups.head(20))
else:
    print("\nNo same-timestamp status conflicts found.")

print("\nCleaned shape:", df_clean.shape)
display(df_clean.head())

P4-5 ready
{
  "before_rows": 154,
  "exact_duplicate_rows_removed": 4,
  "after_exact_dedup_rows": 150,
  "same_timestamp_conflict_groups": 0,
  "same_timestamp_conflict_rows_removed": 0,
  "after_dedup_rows": 150,
  "total_rows_removed": 4,
  "same_timestamp_conflict_policy": "drop_conflicts"
}

No same-timestamp status conflicts found.

Cleaned shape: (150, 6)


,occurred,mc_no,mc_status_raw,process,mc_status,mc_status_token
0,2026-04-03 02:13:00,BLDC-CT-02,Alarm Palette,apb_test,alarm_palette,alarm_palette
1,2026-04-03 23:10:00,BLDC-CT-02,Alarm Coating,apb_test,alarm_coating,alarm_coating
2,2026-04-04 04:10:00,BLDC-CT-02,Alarm Other,apb_test,alarm_other,alarm_other
3,2026-04-05 07:20:00,BLDC-CT-02,Alarm Coating,apb_test,alarm_coating,alarm_coating
4,2026-04-05 13:26:00,BLDC-CT-02,Alarm Other,apb_test,alarm_other,alarm_other


P4-6 — Dataset profiling and validation report

In [7]:
# ============================================================
# P4-6 — Dataset profiling and validation report
# ============================================================

def add_check(checks, name, passed, detail, severity="ERROR"):
    checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })


def build_dataset_profile(df: pd.DataFrame) -> dict:
    return {
        "rows": int(len(df)),
        "columns": list(df.columns),
        "process_count": int(df["process"].nunique()),
        "process_values": sorted(df["process"].dropna().unique().tolist()),
        "machine_count": int(df["mc_no"].nunique()),
        "status_count": int(df["mc_status"].nunique()),
        "time_min": df["occurred"].min(),
        "time_max": df["occurred"].max(),
        "time_span": df["occurred"].max() - df["occurred"].min(),
        "missing_occurred": int(df["occurred"].isna().sum()),
        "missing_mc_no": int(df["mc_no"].isna().sum()),
        "missing_mc_status": int(df["mc_status"].isna().sum()),
        "missing_process": int(df["process"].isna().sum()),
    }


profile = build_dataset_profile(df_clean)

status_profile = (
    df_clean["mc_status"]
    .value_counts(dropna=False)
    .rename_axis("mc_status")
    .reset_index(name="count")
)

status_profile["role"] = status_profile["mc_status"].map(STATUS_ROLE).fillna("unknown")
status_profile["percent"] = status_profile["count"] / len(df_clean) * 100
status_profile = status_profile[["mc_status", "role", "count", "percent"]]

machine_profile = (
    df_clean["mc_no"]
    .value_counts()
    .rename_axis("mc_no")
    .reset_index(name="rows")
)

process_profile = (
    df_clean["process"]
    .value_counts()
    .rename_axis("process")
    .reset_index(name="rows")
)

# Remaining duplicate/conflict checks after dedup
exact_key = ["process", "mc_no", "occurred", "mc_status"]
timestamp_key = ["process", "mc_no", "occurred"]

remaining_exact_duplicates = int(df_clean.duplicated(subset=exact_key).sum())

remaining_timestamp_conflicts = (
    df_clean.groupby(timestamp_key)["mc_status"]
    .nunique()
    .reset_index(name="n_status")
)
remaining_timestamp_conflicts = int((remaining_timestamp_conflicts["n_status"] > 1).sum())

target_row_count = int(df_clean["mc_status"].isin(TARGET_STATUSES).sum())
normal_row_count = int(df_clean["mc_status"].isin(NORMAL_STATUSES).sum())
ignored_row_count = int(df_clean["mc_status"].isin(IGNORED_STATUSES).sum())

checks = []

add_check(
    checks,
    "required_columns_exist",
    all(c in df_clean.columns for c in ["occurred", "mc_no", "mc_status", "process"]),
    "Required internal columns: occurred, mc_no, mc_status, process",
)

add_check(
    checks,
    "row_count_minimum",
    len(df_clean) >= CFG.min_rows,
    f"rows={len(df_clean):,}, minimum={CFG.min_rows:,}",
)

add_check(
    checks,
    "machine_count_minimum",
    df_clean["mc_no"].nunique() >= CFG.min_machines,
    f"machines={df_clean['mc_no'].nunique():,}, minimum={CFG.min_machines:,}",
)

add_check(
    checks,
    "single_process_dataset",
    CFG.allow_multiple_processes or df_clean["process"].nunique() == 1,
    f"process_count={df_clean['process'].nunique()}, processes={sorted(df_clean['process'].unique().tolist())}",
)

add_check(
    checks,
    "no_missing_occurred",
    df_clean["occurred"].notna().all(),
    f"missing_occurred={df_clean['occurred'].isna().sum():,}",
)

add_check(
    checks,
    "no_missing_mc_no",
    df_clean["mc_no"].notna().all() and (df_clean["mc_no"].astype(str).str.strip() != "").all(),
    f"missing_or_blank_mc_no={int(df_clean['mc_no'].isna().sum() + (df_clean['mc_no'].astype(str).str.strip() == '').sum()):,}",
)

add_check(
    checks,
    "no_missing_mc_status",
    df_clean["mc_status"].notna().all() and (df_clean["mc_status"].astype(str).str.strip() != "").all(),
    f"missing_or_blank_mc_status={int(df_clean['mc_status'].isna().sum() + (df_clean['mc_status'].astype(str).str.strip() == '').sum()):,}",
)

add_check(
    checks,
    "normal_statuses_defined",
    len(NORMAL_STATUSES) > 0,
    f"normal_statuses={sorted(NORMAL_STATUSES)}",
)

add_check(
    checks,
    "target_statuses_defined",
    len(TARGET_STATUSES) > 0,
    f"target_statuses={sorted(TARGET_STATUSES)}",
)

add_check(
    checks,
    "target_rows_minimum",
    target_row_count >= CFG.min_target_rows,
    f"target_rows={target_row_count:,}, minimum={CFG.min_target_rows:,}",
)

add_check(
    checks,
    "no_remaining_exact_duplicates",
    remaining_exact_duplicates == 0,
    f"remaining_exact_duplicates={remaining_exact_duplicates:,}",
)

add_check(
    checks,
    "no_remaining_same_timestamp_conflicts",
    remaining_timestamp_conflicts == 0,
    f"remaining_same_timestamp_conflict_groups={remaining_timestamp_conflicts:,}",
    severity="WARNING" if CFG.same_timestamp_conflict_policy == "keep" else "ERROR",
)

configured_normal_missing = sorted(NORMAL_STATUSES - set(ALL_STATUSES))
add_check(
    checks,
    "configured_normal_statuses_present",
    len(configured_normal_missing) == 0,
    f"configured normal statuses not present in data: {configured_normal_missing}",
    severity="WARNING",
)

validation_df = pd.DataFrame(checks)

failed_errors = validation_df[
    (validation_df["severity"] == "ERROR") & (~validation_df["passed"])
]

READY_FOR_BATCH_2 = len(failed_errors) == 0

print("=" * 100)
print("P4-6 — DATASET PROFILE")
print("=" * 100)

print(f"Pipeline:        {CFG.pipeline_name}")
print(f"Process ID:      {CFG.process_id}")
print(f"Data path:       {CFG.data_path}")
print(f"Rows:            {profile['rows']:,}")
print(f"Machines:        {profile['machine_count']:,}")
print(f"Statuses:        {profile['status_count']:,}")
print(f"Processes:       {profile['process_values']}")
print(f"Time range:      {profile['time_min']} -> {profile['time_max']}")
print(f"Time span:       {profile['time_span']}")

print("\nNormal statuses:")
print(sorted(NORMAL_STATUSES))

print("\nTarget statuses:")
print(sorted(TARGET_STATUSES))

print("\nIgnored statuses:")
print(sorted(IGNORED_STATUSES))

print("\nDeduplication report:")
print(json.dumps(dedup_report, indent=2, ensure_ascii=False))

print("\nStatus profile:")
display(status_profile)

print("\nProcess profile:")
display(process_profile)

print("\nMachine row-count summary:")
display(machine_profile["rows"].describe().to_frame("rows"))

print("\nTop 20 machines by row count:")
display(machine_profile.head(20))

print("\nValidation gates:")
display(validation_df)

print("=" * 100)
if READY_FOR_BATCH_2:
    print("READY_FOR_BATCH_2 = True")
    print("Batch 1 passed. Send this output to confirm before moving to Batch 2.")
else:
    print("READY_FOR_BATCH_2 = False")
    print("Batch 1 has failed ERROR gates. Fix before moving to Batch 2.")
print("=" * 100)

P4-6 — DATASET PROFILE
Pipeline:        automatic_model_training_pipeline
Process ID:      apb
Data path:       apb_data_cleaned.xlsx
Rows:            150
Machines:        2
Statuses:        3
Processes:       ['apb_test']
Time range:      2026-04-03 00:05:00 -> 2026-06-16 04:45:00
Time span:       74 days 04:40:00

Normal statuses:
['alarm_other']

Target statuses:
['alarm_coating', 'alarm_palette']

Ignored statuses:
[]

Deduplication report:
{
  "before_rows": 154,
  "exact_duplicate_rows_removed": 4,
  "after_exact_dedup_rows": 150,
  "same_timestamp_conflict_groups": 0,
  "same_timestamp_conflict_rows_removed": 0,
  "after_dedup_rows": 150,
  "total_rows_removed": 4,
  "same_timestamp_conflict_policy": "drop_conflicts"
}

Status profile:


,mc_status,role,count,percent
0,alarm_other,normal,63,42.000000
1,alarm_coating,target,46,30.666667
2,alarm_palette,target,41,27.333333



Process profile:


,process,rows
0,apb_test,150



Machine row-count summary:


...


Top 20 machines by row count:


,mc_no,rows
0,BLDC-CT-02,120
1,BLDC-CT-03,30



Validation gates:


,check,severity,passed,detail
0,required_columns_exist,ERROR,True,"Required internal columns: occurred, mc_no, mc..."
1,row_count_minimum,ERROR,False,"rows=150, minimum=1,000"
2,machine_count_minimum,ERROR,True,"machines=2, minimum=1"
3,single_process_dataset,ERROR,True,"process_count=1, processes=['apb_test']"
4,no_missing_occurred,ERROR,True,missing_occurred=0
5,no_missing_mc_no,ERROR,True,missing_or_blank_mc_no=0
6,no_missing_mc_status,ERROR,True,missing_or_blank_mc_status=0
7,normal_statuses_defined,ERROR,True,normal_statuses=['alarm_other']
8,target_statuses_defined,ERROR,True,"target_statuses=['alarm_coating', 'alarm_palet..."
9,target_rows_minimum,ERROR,False,"target_rows=87, minimum=100"


READY_FOR_BATCH_2 = False
Batch 1 has failed ERROR gates. Fix before moving to Batch 2.


P4-7 — Build next-target-event labels

This cell creates the prediction target:
- next_target_time
- next_target_type
- y_time_sec

The target means:

For each machine event row, find the next future event where mc_status is in TARGET_STATUSES.

In [8]:
# ============================================================
# P4-7 — Build next-target-event labels
# ============================================================

def build_next_target_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Build next-target-event labels per process + machine.

    Target event definition:
      mc_status in TARGET_STATUSES

    For every row:
      next_target_time = next future timestamp where status is target
      next_target_type = status of that next target event
      y_time_sec       = seconds until next target event

    Important:
      If the current row is already a target event, the label is still the NEXT
      future target, not the current row itself.
    """
    out = df.copy()

    out["is_normal_status"] = out["mc_status"].isin(NORMAL_STATUSES).astype(int)
    out["is_target_event"] = out["mc_status"].isin(TARGET_STATUSES).astype(int)
    out["is_ignored_status"] = out["mc_status"].isin(IGNORED_STATUSES).astype(int)

    out = out.sort_values(["process", "mc_no", "occurred", "mc_status"]).reset_index(
        drop=True
    )

    def _label_one_machine(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values(["occurred", "mc_status"]).copy()

        target_time_at_row = g["occurred"].where(g["is_target_event"].eq(1))
        target_type_at_row = g["mc_status"].where(g["is_target_event"].eq(1))

        # shift(-1) makes the label strictly future.
        # bfill fills normal/ignored rows with the next future target.
        g["next_target_time"] = target_time_at_row.shift(-1).bfill()
        g["next_target_type"] = target_type_at_row.shift(-1).bfill()

        g["y_time_sec"] = (
            g["next_target_time"] - g["occurred"]
        ).dt.total_seconds()

        return g

    # --- FIX APPLIED HERE ---
    # 1. Remove group_keys=False so keys are preserved in the temporary MultiIndex
    # 2. Use reset_index(level=...) to safely pull them back into regular columns
    out = (
        out.groupby(["process", "mc_no"])
        .apply(_label_one_machine)
        .reset_index(level=["process", "mc_no"])
        .reset_index(drop=True)
    )

    return out


df_label_raw = build_next_target_labels(df_clean)

print("P4-7 ready")
print("Input rows:", f"{len(df_clean):,}")
print("Rows after raw labeling:", f"{len(df_label_raw):,}")

display(df_label_raw.head(20))

print("\nLabel columns:")
display(
    df_label_raw[
        [
            "process",
            "mc_no",
            "occurred",
            "mc_status",
            "is_normal_status",
            "is_target_event",
            "next_target_time",
            "next_target_type",
            "y_time_sec",
        ]
    ].head(30)
)

P4-7 ready
Input rows: 150
Rows after raw labeling: 150


,process,mc_no,occurred,mc_status_raw,mc_status,mc_status_token,is_normal_status,is_target_event,is_ignored_status,next_target_time,next_target_type,y_time_sec
0,apb_test,BLDC-CT-02,2026-04-03 02:13:00,Alarm Palette,alarm_palette,alarm_palette,0,1,0,2026-04-03 23:10:00,alarm_coating,75420.0
1,apb_test,BLDC-CT-02,2026-04-03 23:10:00,Alarm Coating,alarm_coating,alarm_coating,0,1,0,2026-04-05 07:20:00,alarm_coating,115800.0
2,apb_test,BLDC-CT-02,2026-04-04 04:10:00,Alarm Other,alarm_other,alarm_other,1,0,0,2026-04-05 07:20:00,alarm_coating,97800.0
3,apb_test,BLDC-CT-02,2026-04-05 07:20:00,Alarm Coating,alarm_coating,alarm_coating,0,1,0,2026-04-06 15:48:00,alarm_palette,116880.0
4,apb_test,BLDC-CT-02,2026-04-05 13:26:00,Alarm Other,alarm_other,alarm_other,1,0,0,2026-04-06 15:48:00,alarm_palette,94920.0
5,apb_test,BLDC-CT-02,2026-04-06 15:48:00,Alarm Palette,alarm_palette,alarm_palette,0,1,0,2026-04-08 13:21:00,alarm_palette,163980.0
6,apb_test,BLDC-CT-02,2026-04-07 08:50:00,Alarm Other,alarm_other,alarm_other,1,0,0,2026-04-08 13:21:00,alarm_palette,102660.0
7,apb_test,BLDC-CT-02,2026-04-07 20:30:00,Alarm Other,alarm_other,alarm_other,1,0,0,2026-04-08 13:21:00,alarm_palette,60660.0
8,apb_test,BLDC-CT-02,2026-04-08 12:15:00,Alarm Other,alarm_other,alarm_other,1,0,0,2026-04-08 13:21:00,alarm_palette,3960.0
9,apb_test,BLDC-CT-02,2026-04-08 13:21:00,Alarm Palette,alarm_palette,alarm_palette,0,1,0,2026-04-09 11:15:00,alarm_coating,78840.0



Label columns:


,process,mc_no,occurred,mc_status,is_normal_status,is_target_event,next_target_time,next_target_type,y_time_sec
0,apb_test,BLDC-CT-02,2026-04-03 02:13:00,alarm_palette,0,1,2026-04-03 23:10:00,alarm_coating,75420.0
1,apb_test,BLDC-CT-02,2026-04-03 23:10:00,alarm_coating,0,1,2026-04-05 07:20:00,alarm_coating,115800.0
2,apb_test,BLDC-CT-02,2026-04-04 04:10:00,alarm_other,1,0,2026-04-05 07:20:00,alarm_coating,97800.0
3,apb_test,BLDC-CT-02,2026-04-05 07:20:00,alarm_coating,0,1,2026-04-06 15:48:00,alarm_palette,116880.0
4,apb_test,BLDC-CT-02,2026-04-05 13:26:00,alarm_other,1,0,2026-04-06 15:48:00,alarm_palette,94920.0
5,apb_test,BLDC-CT-02,2026-04-06 15:48:00,alarm_palette,0,1,2026-04-08 13:21:00,alarm_palette,163980.0
6,apb_test,BLDC-CT-02,2026-04-07 08:50:00,alarm_other,1,0,2026-04-08 13:21:00,alarm_palette,102660.0
7,apb_test,BLDC-CT-02,2026-04-07 20:30:00,alarm_other,1,0,2026-04-08 13:21:00,alarm_palette,60660.0
8,apb_test,BLDC-CT-02,2026-04-08 12:15:00,alarm_other,1,0,2026-04-08 13:21:00,alarm_palette,3960.0
9,apb_test,BLDC-CT-02,2026-04-08 13:21:00,alarm_palette,0,1,2026-04-09 11:15:00,alarm_coating,78840.0


P4-8 — Remove invalid target rows

This removes rows where the machine has no future target event, or where the computed target gap is invalid.

In [9]:
# ============================================================
# P4-8 — Remove invalid target rows
# ============================================================

def clean_labeled_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    before = len(df)

    missing_next_time = int(df["next_target_time"].isna().sum())
    missing_next_type = int(df["next_target_type"].isna().sum())
    missing_y = int(df["y_time_sec"].isna().sum())
    non_positive_y = int((df["y_time_sec"] <= 0).sum())

    # Keep only rows with a valid strictly-future target.
    out = df.copy()
    out = out.dropna(subset=["next_target_time", "next_target_type", "y_time_sec"])
    out = out[out["y_time_sec"] > 0].copy()

    after = len(out)

    report = {
        "before_rows": before,
        "missing_next_target_time_rows": missing_next_time,
        "missing_next_target_type_rows": missing_next_type,
        "missing_y_time_rows": missing_y,
        "non_positive_y_time_rows": non_positive_y,
        "after_valid_label_rows": after,
        "rows_removed": before - after,
        "rows_removed_percent": (before - after) / before * 100 if before else np.nan,
    }

    out = out.sort_values(["process", "mc_no", "occurred", "mc_status"]).reset_index(drop=True)

    return out, report


df_labeled, label_cleaning_report = clean_labeled_rows(df_label_raw)

print("P4-8 ready")
print(json.dumps(label_cleaning_report, indent=2, ensure_ascii=False))

print("\nLabeled shape:", df_labeled.shape)

display(
    df_labeled[
        [
            "process",
            "mc_no",
            "occurred",
            "mc_status",
            "is_normal_status",
            "is_target_event",
            "next_target_time",
            "next_target_type",
            "y_time_sec",
        ]
    ].head(30)
)

P4-8 ready
{
  "before_rows": 150,
  "missing_next_target_time_rows": 5,
  "missing_next_target_type_rows": 5,
  "missing_y_time_rows": 5,
  "non_positive_y_time_rows": 0,
  "after_valid_label_rows": 145,
  "rows_removed": 5,
  "rows_removed_percent": 3.3333333333333335
}

Labeled shape: (145, 12)


,process,mc_no,occurred,mc_status,is_normal_status,is_target_event,next_target_time,next_target_type,y_time_sec
0,apb_test,BLDC-CT-02,2026-04-03 02:13:00,alarm_palette,0,1,2026-04-03 23:10:00,alarm_coating,75420.0
1,apb_test,BLDC-CT-02,2026-04-03 23:10:00,alarm_coating,0,1,2026-04-05 07:20:00,alarm_coating,115800.0
2,apb_test,BLDC-CT-02,2026-04-04 04:10:00,alarm_other,1,0,2026-04-05 07:20:00,alarm_coating,97800.0
3,apb_test,BLDC-CT-02,2026-04-05 07:20:00,alarm_coating,0,1,2026-04-06 15:48:00,alarm_palette,116880.0
4,apb_test,BLDC-CT-02,2026-04-05 13:26:00,alarm_other,1,0,2026-04-06 15:48:00,alarm_palette,94920.0
5,apb_test,BLDC-CT-02,2026-04-06 15:48:00,alarm_palette,0,1,2026-04-08 13:21:00,alarm_palette,163980.0
6,apb_test,BLDC-CT-02,2026-04-07 08:50:00,alarm_other,1,0,2026-04-08 13:21:00,alarm_palette,102660.0
7,apb_test,BLDC-CT-02,2026-04-07 20:30:00,alarm_other,1,0,2026-04-08 13:21:00,alarm_palette,60660.0
8,apb_test,BLDC-CT-02,2026-04-08 12:15:00,alarm_other,1,0,2026-04-08 13:21:00,alarm_palette,3960.0
9,apb_test,BLDC-CT-02,2026-04-08 13:21:00,alarm_palette,0,1,2026-04-09 11:15:00,alarm_coating,78840.0


P4-9 — Target distribution and label validation report

In [10]:
# ============================================================
# P4-9 — Target distribution and label validation report
# ============================================================

def seconds_to_readable(sec):
    if pd.isna(sec):
        return np.nan

    sec = float(sec)

    if sec < 60:
        return f"{sec:.1f}s"
    elif sec < 3600:
        return f"{sec / 60:.2f}m"
    elif sec < 86400:
        return f"{sec / 3600:.2f}h"
    else:
        return f"{sec / 86400:.2f}d"


def target_distribution_table(y: pd.Series) -> pd.DataFrame:
    qs = {
        "min": y.min(),
        "p01": y.quantile(0.01),
        "p05": y.quantile(0.05),
        "p10": y.quantile(0.10),
        "p25": y.quantile(0.25),
        "p50_median": y.quantile(0.50),
        "p75": y.quantile(0.75),
        "p90": y.quantile(0.90),
        "p95": y.quantile(0.95),
        "p99": y.quantile(0.99),
        "max": y.max(),
        "mean": y.mean(),
        "std": y.std(),
    }

    out = pd.DataFrame({
        "metric": list(qs.keys()),
        "seconds": list(qs.values()),
    })

    out["readable"] = out["seconds"].apply(seconds_to_readable)
    return out


def build_label_validation_report(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add(name, passed, detail, severity="ERROR"):
        checks.append({
            "check": name,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        })

    add(
        "labeled_rows_exist",
        len(df) > 0,
        f"labeled_rows={len(df):,}",
    )

    add(
        "target_status_rows_exist",
        int(df["is_target_event"].sum()) >= CFG.min_target_rows,
        f"current_row_target_events={int(df['is_target_event'].sum()):,}, minimum={CFG.min_target_rows:,}",
    )

    add(
        "next_target_type_has_no_missing",
        df["next_target_type"].notna().all(),
        f"missing_next_target_type={df['next_target_type'].isna().sum():,}",
    )

    add(
        "next_target_time_has_no_missing",
        df["next_target_time"].notna().all(),
        f"missing_next_target_time={df['next_target_time'].isna().sum():,}",
    )

    add(
        "y_time_has_no_missing",
        df["y_time_sec"].notna().all(),
        f"missing_y_time_sec={df['y_time_sec'].isna().sum():,}",
    )

    add(
        "y_time_strictly_positive",
        (df["y_time_sec"] > 0).all(),
        f"non_positive_y_time_rows={int((df['y_time_sec'] <= 0).sum()):,}",
    )

    normal_in_next_target = sorted(set(df["next_target_type"].unique()) & NORMAL_STATUSES)
    add(
        "normal_statuses_not_in_next_target_type",
        len(normal_in_next_target) == 0,
        f"normal statuses found in next_target_type: {normal_in_next_target}",
    )

    ignored_in_next_target = sorted(set(df["next_target_type"].unique()) & IGNORED_STATUSES)
    add(
        "ignored_statuses_not_in_next_target_type",
        len(ignored_in_next_target) == 0,
        f"ignored statuses found in next_target_type: {ignored_in_next_target}",
    )

    unexpected_next_targets = sorted(set(df["next_target_type"].unique()) - TARGET_STATUSES)
    add(
        "next_target_type_only_contains_target_statuses",
        len(unexpected_next_targets) == 0,
        f"unexpected next target types: {unexpected_next_targets}",
    )

    add(
        "next_target_time_is_future",
        (df["next_target_time"] > df["occurred"]).all(),
        f"rows_where_next_target_not_future={int((df['next_target_time'] <= df['occurred']).sum()):,}",
    )

    add(
        "multiple_target_classes",
        df["next_target_type"].nunique() >= 2,
        f"next_target_type_classes={df['next_target_type'].nunique():,}",
        severity="WARNING",
    )

    return pd.DataFrame(checks)


target_time_dist = target_distribution_table(df_labeled["y_time_sec"])

next_target_type_dist = (
    df_labeled["next_target_type"]
    .value_counts(dropna=False)
    .rename_axis("next_target_type")
    .reset_index(name="count")
)

next_target_type_dist["percent"] = (
    next_target_type_dist["count"] / len(df_labeled) * 100
)

current_status_dist_after_label = (
    df_labeled["mc_status"]
    .value_counts(dropna=False)
    .rename_axis("current_mc_status")
    .reset_index(name="count")
)

current_status_dist_after_label["role"] = current_status_dist_after_label["current_mc_status"].map(STATUS_ROLE)
current_status_dist_after_label["percent"] = (
    current_status_dist_after_label["count"] / len(df_labeled) * 100
)

machine_label_summary = (
    df_labeled.groupby(["process", "mc_no"])
    .agg(
        rows=("mc_no", "size"),
        current_target_rows=("is_target_event", "sum"),
        first_time=("occurred", "min"),
        last_time=("occurred", "max"),
        median_y_sec=("y_time_sec", "median"),
        p90_y_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        max_y_sec=("y_time_sec", "max"),
        next_target_classes=("next_target_type", "nunique"),
    )
    .reset_index()
)

machine_label_row_summary = machine_label_summary["rows"].describe().to_frame("labeled_rows")
machine_target_row_summary = machine_label_summary["current_target_rows"].describe().to_frame("current_target_rows")

low_labeled_machines = machine_label_summary[machine_label_summary["rows"] < 50].sort_values("rows")
zero_current_target_machines = machine_label_summary[machine_label_summary["current_target_rows"] == 0]

label_validation_df = build_label_validation_report(df_labeled)

failed_label_errors = label_validation_df[
    (label_validation_df["severity"] == "ERROR") & (~label_validation_df["passed"])
]

READY_FOR_BATCH_3 = len(failed_label_errors) == 0

print("=" * 100)
print("P4-9 — TARGET DISTRIBUTION AND LABEL VALIDATION REPORT")
print("=" * 100)

print(f"Rows before labeling:       {len(df_clean):,}")
print(f"Rows after valid labeling:  {len(df_labeled):,}")
print(f"Rows removed:               {label_cleaning_report['rows_removed']:,}")
print(f"Rows removed percent:       {label_cleaning_report['rows_removed_percent']:.4f}%")

print("\nNormal statuses:")
print(sorted(NORMAL_STATUSES))

print("\nTarget statuses:")
print(sorted(TARGET_STATUSES))

print("\nLabel cleaning report:")
print(json.dumps(label_cleaning_report, indent=2, ensure_ascii=False))

print("\nTarget time distribution:")
display(target_time_dist)

print("\nNext target type distribution:")
display(next_target_type_dist)

print("\nCurrent status distribution after valid labeling:")
display(current_status_dist_after_label)

print("\nMachine labeled row-count summary:")
display(machine_label_row_summary)

print("\nMachine current-target-row summary:")
display(machine_target_row_summary)

print("\nLowest 20 machines by labeled rows:")
display(low_labeled_machines.head(20))

print("\nMachines with zero current target rows after labeling:")
display(zero_current_target_machines.head(20))

print("\nLabel validation gates:")
display(label_validation_df)

print("=" * 100)
if READY_FOR_BATCH_3:
    print("READY_FOR_BATCH_3 = True")
    print("Batch 2 passed. Send this output to confirm before moving to Batch 3.")
else:
    print("READY_FOR_BATCH_3 = False")
    print("Batch 2 has failed ERROR gates. Fix before moving to Batch 3.")
print("=" * 100)

P4-9 — TARGET DISTRIBUTION AND LABEL VALIDATION REPORT
Rows before labeling:       150
Rows after valid labeling:  145
Rows removed:               5
Rows removed percent:       3.3333%

Normal statuses:
['alarm_other']

Target statuses:
['alarm_coating', 'alarm_palette']

Label cleaning report:
{
  "before_rows": 150,
  "missing_next_target_time_rows": 5,
  "missing_next_target_type_rows": 5,
  "missing_y_time_rows": 5,
  "non_positive_y_time_rows": 0,
  "after_valid_label_rows": 145,
  "rows_removed": 5,
  "rows_removed_percent": 3.3333333333333335
}

Target time distribution:


,metric,seconds,readable
0,min,1800.000000,30.00m
1,p01,3086.400000,51.44m
2,p05,5808.000000,1.61h
3,p10,8760.000000,2.43h
4,p25,29700.000000,8.25h
5,p50_median,71820.000000,19.95h
6,p75,178320.000000,2.06d
7,p90,432180.000000,5.00d
8,p95,505836.000000,5.85d
9,p99,754536.000000,8.73d



Next target type distribution:


,next_target_type,count,percent
0,alarm_palette,73,50.344828
1,alarm_coating,72,49.655172



Current status distribution after valid labeling:


...


Machines with zero current target rows after labeling:


,process,mc_no,rows,current_target_rows,first_time,last_time,median_y_sec,p90_y_sec,max_y_sec,next_target_classes



Label validation gates:


,check,severity,passed,detail
0,labeled_rows_exist,ERROR,True,labeled_rows=145
1,target_status_rows_exist,ERROR,False,"current_row_target_events=85, minimum=100"
2,next_target_type_has_no_missing,ERROR,True,missing_next_target_type=0
3,next_target_time_has_no_missing,ERROR,True,missing_next_target_time=0
4,y_time_has_no_missing,ERROR,True,missing_y_time_sec=0
5,y_time_strictly_positive,ERROR,True,non_positive_y_time_rows=0
6,normal_statuses_not_in_next_target_type,ERROR,True,normal statuses found in next_target_type: []
7,ignored_statuses_not_in_next_target_type,ERROR,True,ignored statuses found in next_target_type: []
8,next_target_type_only_contains_target_statuses,ERROR,True,unexpected next target types: []
9,next_target_time_is_future,ERROR,True,rows_where_next_target_not_future=0


READY_FOR_BATCH_3 = False
Batch 2 has failed ERROR gates. Fix before moving to Batch 3.


P4-10 — Build automatic feature set

In [11]:
# ============================================================
# P4-10 — Build automatic feature set
# ============================================================

def safe_seconds_delta(a, b):
    return (a - b).dt.total_seconds()


def make_one_hot_from_token(series: pd.Series, prefix: str) -> pd.DataFrame:
    """
    Stable one-hot helper.
    Example:
      status_run
      status_mc_alarm
      prev1_status_mc_waitpart
    """
    d = pd.get_dummies(series.fillna("none").astype(str), prefix=prefix, dtype=np.int8)
    d.columns = [sanitize_feature_token(c) for c in d.columns]
    return d


def add_groupwise_last_time(out: pd.DataFrame, condition: pd.Series, col_name: str, include_current: bool = True):
    """
    Create last timestamp where condition was true within each process+machine group.

    include_current=True:
      current row is allowed to update last timestamp.
      Good for time_since_each_status.

    include_current=False:
      current row is excluded.
      Good for time_since_last_target_event before current row.
    """
    temp_time = out["occurred"].where(condition)

    if not include_current:
        temp_time = temp_time.groupby([out["process"], out["mc_no"]]).shift(1)

    out[col_name] = temp_time.groupby([out["process"], out["mc_no"]]).ffill()
    return out


def add_rolling_count_features(
    out: pd.DataFrame,
    value_col: str,
    prefix: str,
    windows_min: list,
) -> pd.DataFrame:
    """
    Time-based rolling sums per process+machine.

    value_col should be numeric 0/1.
    Rolling includes the current row, which is valid because current event is already observed.
    """
    base = out[["process", "mc_no", "occurred", value_col]].copy()
    base = base.sort_values(["process", "mc_no", "occurred"])

    for w in windows_min:
        feature_count = f"{prefix}_{w}m_count"
        feature_rate = f"{prefix}_{w}m_rate_per_min"

        rolled = (
            base
            .set_index("occurred")
            .groupby(["process", "mc_no"])[value_col]
            .rolling(f"{w}min", min_periods=1)
            .sum()
            .reset_index(level=[0, 1], drop=True)
        )

        out[feature_count] = rolled.reindex(out.set_index("occurred").index).to_numpy()

        # Since index reindex by occurred alone can be unsafe when timestamps repeat across machines,
        # recalculate by aligned sorted order instead.
        # The block above is kept readable, but the aligned implementation below is the one used.
        rolled_aligned = (
            base
            .set_index("occurred")
            .groupby(["process", "mc_no"])[value_col]
            .rolling(f"{w}min", min_periods=1)
            .sum()
            .reset_index()
        )

        out[feature_count] = rolled_aligned[value_col].to_numpy()
        out[feature_rate] = out[feature_count] / max(w, 1)

    return out


def build_automatic_features(df: pd.DataFrame, cfg: AutoTrainingConfig) -> tuple[pd.DataFrame, dict]:
    """
    Build process-agnostic effect-only features.

    Uses only:
      occurred
      mc_no
      process
      current/past mc_status
      current/past target-event markers

    Does not use:
      next_target_time
      next_target_type
      y_time_sec
    except those remain in dataframe as labels, not features.
    """
    out = df.copy()
    out = out.sort_values(["process", "mc_no", "occurred", "mc_status"]).reset_index(drop=True)

    group_keys = ["process", "mc_no"]
    g = out.groupby(group_keys, group_keys=False)

    # ------------------------------------------------------------
    # Base temporal features
    # ------------------------------------------------------------
    out["hour"] = out["occurred"].dt.hour.astype(np.int16)
    out["minute"] = out["occurred"].dt.minute.astype(np.int16)
    out["second"] = out["occurred"].dt.second.astype(np.int16)
    out["weekday"] = out["occurred"].dt.weekday.astype(np.int16)
    out["day"] = out["occurred"].dt.day.astype(np.int16)
    out["is_weekend"] = out["weekday"].isin([5, 6]).astype(np.int8)
    out["is_night"] = out["hour"].isin([0, 1, 2, 3, 4, 5]).astype(np.int8)

    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["minute_sin"] = np.sin(2 * np.pi * out["minute"] / 60)
    out["minute_cos"] = np.cos(2 * np.pi * out["minute"] / 60)
    out["weekday_sin"] = np.sin(2 * np.pi * out["weekday"] / 7)
    out["weekday_cos"] = np.cos(2 * np.pi * out["weekday"] / 7)

    # ------------------------------------------------------------
    # Machine sequence / age features
    # ------------------------------------------------------------
    out["machine_event_index"] = g.cumcount().astype(np.int32)

    first_seen = g["occurred"].transform("min")
    out["time_since_first_seen_machine_sec"] = safe_seconds_delta(out["occurred"], first_seen)

    # ------------------------------------------------------------
    # Previous event gap features
    # ------------------------------------------------------------
    out["prev_event_time"] = g["occurred"].shift(1)
    out["event_gap_sec"] = safe_seconds_delta(out["occurred"], out["prev_event_time"])

    for lag in [1, 2, 3]:
        out[f"event_gap_lag{lag}_sec"] = g["event_gap_sec"].shift(lag)

    out["event_gap_mean_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_std_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_median_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .median()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_min_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .min()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_max_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .max()
        .reset_index(level=[0, 1], drop=True)
    )

    # ------------------------------------------------------------
    # Current status one-hot
    # ------------------------------------------------------------
    status_ohe = make_one_hot_from_token(out["mc_status_token"], "status")
    out = pd.concat([out, status_ohe], axis=1)

    status_cols = list(status_ohe.columns)

    # ------------------------------------------------------------
    # Previous status sequence features
    # ------------------------------------------------------------
    prev_status_cols = []

    for lag in [1, 2, 3]:
        prev_token = g["mc_status_token"].shift(lag).fillna("none")
        prev_ohe = make_one_hot_from_token(prev_token, f"prev{lag}_status")
        out = pd.concat([out, prev_ohe], axis=1)
        prev_status_cols.extend(prev_ohe.columns.tolist())

    # ------------------------------------------------------------
    # Last target-event type before current row
    # ------------------------------------------------------------
    target_token = out["mc_status_token"].where(out["is_target_event"].eq(1))
    out["last_target_type_token"] = (
        target_token
        .groupby([out["process"], out["mc_no"]])
        .shift(1)
        .groupby([out["process"], out["mc_no"]])
        .ffill()
        .fillna("none")
    )

    last_target_ohe = make_one_hot_from_token(out["last_target_type_token"], "last_target")
    out = pd.concat([out, last_target_ohe], axis=1)
    last_target_cols = list(last_target_ohe.columns)

    # Last target time before current row
    target_time = out["occurred"].where(out["is_target_event"].eq(1))
    out["last_target_time"] = (
        target_time
        .groupby([out["process"], out["mc_no"]])
        .shift(1)
        .groupby([out["process"], out["mc_no"]])
        .ffill()
    )

    out["time_since_last_target_sec"] = safe_seconds_delta(out["occurred"], out["last_target_time"])

    # Number of target events before current row
    out["target_event_count_prior"] = (
        out.groupby(group_keys)["is_target_event"]
        .cumsum()
        .groupby([out["process"], out["mc_no"]])
        .shift(1)
        .fillna(0)
        .astype(np.int32)
    )

    # ------------------------------------------------------------
    # Time since status change / consecutive same status
    # ------------------------------------------------------------
    out["prev_status_token"] = g["mc_status_token"].shift(1)

    out["status_changed"] = (
        out["mc_status_token"].ne(out["prev_status_token"])
        | out["prev_status_token"].isna()
    ).astype(np.int8)

    out["status_segment_id"] = (
        out.groupby(group_keys)["status_changed"]
        .cumsum()
        .astype(np.int32)
    )

    seg_keys = ["process", "mc_no", "status_segment_id"]

    out["status_segment_start_time"] = (
        out.groupby(seg_keys)["occurred"]
        .transform("min")
    )

    out["time_since_status_change_sec"] = safe_seconds_delta(
        out["occurred"],
        out["status_segment_start_time"]
    )

    out["consecutive_same_status_count"] = (
        out.groupby(seg_keys)
        .cumcount()
        .astype(np.int32)
    )

    # ------------------------------------------------------------
    # Time since each canonical status
    # ------------------------------------------------------------
    time_since_status_cols = []

    for status in ALL_STATUSES:
        token = sanitize_feature_token(status)
        col_last = f"_last_time_status_{token}"
        col_feat = f"time_since_status_{token}_sec"

        status_condition = out["mc_status"].eq(status)

        out[col_last] = (
            out["occurred"]
            .where(status_condition)
            .groupby([out["process"], out["mc_no"]])
            .ffill()
        )

        out[col_feat] = safe_seconds_delta(out["occurred"], out[col_last])
        time_since_status_cols.append(col_feat)

    # ------------------------------------------------------------
    # Rolling event / target / status-specific counts and rates
    # ------------------------------------------------------------
    out["_event_one"] = 1
    out["_target_one"] = out["is_target_event"].astype(int)

    rolling_cols = []

    base_sorted_index = out.index

    # Generic event and target rolling features
    for value_col, prefix in [
        ("_event_one", "events"),
        ("_target_one", "targets"),
    ]:
        base = out[["process", "mc_no", "occurred", value_col]].copy()

        for w in cfg.rolling_windows_min:
            count_col = f"{prefix}_{w}m_count"
            rate_col = f"{prefix}_{w}m_rate_per_min"

            rolled = (
                base
                .set_index("occurred")
                .groupby(["process", "mc_no"])[value_col]
                .rolling(f"{w}min", min_periods=1)
                .sum()
                .reset_index()
            )

            out[count_col] = rolled[value_col].to_numpy()
            out[rate_col] = out[count_col] / max(w, 1)

            rolling_cols.extend([count_col, rate_col])

    # Status-specific rolling features
    status_specific_rolling_cols = []

    for status in ALL_STATUSES:
        token = sanitize_feature_token(status)
        temp_col = f"_is_status_{token}"
        out[temp_col] = out["mc_status"].eq(status).astype(int)

        base = out[["process", "mc_no", "occurred", temp_col]].copy()

        for w in cfg.rolling_windows_min:
            count_col = f"status_{token}_{w}m_count"
            rate_col = f"status_{token}_{w}m_rate_per_min"

            rolled = (
                base
                .set_index("occurred")
                .groupby(["process", "mc_no"])[temp_col]
                .rolling(f"{w}min", min_periods=1)
                .sum()
                .reset_index()
            )

            out[count_col] = rolled[temp_col].to_numpy()
            out[rate_col] = out[count_col] / max(w, 1)

            status_specific_rolling_cols.extend([count_col, rate_col])

    rolling_cols.extend(status_specific_rolling_cols)

    # ------------------------------------------------------------
    # Machine-level behavioral features
    # These are initially calculated on all labeled data for feature construction.
    # In Batch 4, train-only maps/fill values will be fitted for final production-safe artifacts.
    # ------------------------------------------------------------
    machine_behavior = (
        out.groupby(["process", "mc_no"])
        .agg(
            mc_rows=("mc_no", "size"),
            mc_target_ratio=("is_target_event", "mean"),
            mc_median_event_gap_sec=("event_gap_sec", "median"),
            mc_median_y_sec=("y_time_sec", "median"),
            mc_p90_y_sec=("y_time_sec", lambda x: x.quantile(0.90)),
            mc_target_classes_seen=("next_target_type", "nunique"),
        )
        .reset_index()
    )

    out = out.merge(machine_behavior, on=["process", "mc_no"], how="left")

    machine_behavior_cols = [
        "mc_rows",
        "mc_target_ratio",
        "mc_median_event_gap_sec",
        "mc_median_y_sec",
        "mc_p90_y_sec",
        "mc_target_classes_seen",
    ]

    # ------------------------------------------------------------
    # Status-level behavioral features
    # ------------------------------------------------------------
    status_behavior = (
        out.groupby("mc_status")
        .agg(
            status_rows=("mc_status", "size"),
            status_target_ratio=("is_target_event", "mean"),
            status_median_y_sec=("y_time_sec", "median"),
            status_p90_y_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        )
        .reset_index()
    )

    out = out.merge(status_behavior, on="mc_status", how="left")

    status_behavior_cols = [
        "status_rows",
        "status_target_ratio",
        "status_median_y_sec",
        "status_p90_y_sec",
    ]

    # ------------------------------------------------------------
    # Normalized timing ratio features
    # ------------------------------------------------------------
    eps = 1.0

    out["time_since_last_target_over_mc_median_y"] = (
        out["time_since_last_target_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["event_gap_over_mc_median_event_gap"] = (
        out["event_gap_sec"] / (out["mc_median_event_gap_sec"] + eps)
    )

    out["time_since_status_change_over_mc_median_y"] = (
        out["time_since_status_change_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["target_event_count_prior_over_machine_events"] = (
        out["target_event_count_prior"] / (out["machine_event_index"] + 1)
    )

    ratio_cols = [
        "time_since_last_target_over_mc_median_y",
        "event_gap_over_mc_median_event_gap",
        "time_since_status_change_over_mc_median_y",
        "target_event_count_prior_over_machine_events",
    ]

    # ------------------------------------------------------------
    # Cleanup helper timestamp columns, keep useful metadata columns
    # ------------------------------------------------------------
    helper_drop_cols = []

    helper_drop_cols.extend([c for c in out.columns if c.startswith("_last_time_status_")])
    helper_drop_cols.extend([c for c in out.columns if c.startswith("_is_status_")])
    helper_drop_cols.extend(["_event_one", "_target_one"])

    out = out.drop(columns=[c for c in helper_drop_cols if c in out.columns])

    feature_groups = {
        "base_time_cols": [
            "hour", "minute", "second", "weekday", "day", "is_weekend", "is_night",
            "hour_sin", "hour_cos", "minute_sin", "minute_cos", "weekday_sin", "weekday_cos",
        ],
        "machine_sequence_cols": [
            "machine_event_index",
            "time_since_first_seen_machine_sec",
        ],
        "event_gap_cols": [
            "event_gap_sec",
            "event_gap_lag1_sec",
            "event_gap_lag2_sec",
            "event_gap_lag3_sec",
            "event_gap_mean_5_sec",
            "event_gap_std_5_sec",
            "event_gap_median_5_sec",
            "event_gap_min_5_sec",
            "event_gap_max_5_sec",
        ],
        "status_cols": status_cols,
        "prev_status_cols": prev_status_cols,
        "last_target_cols": last_target_cols,
        "target_history_cols": [
            "time_since_last_target_sec",
            "target_event_count_prior",
        ],
        "status_duration_cols": [
            "status_changed",
            "time_since_status_change_sec",
            "consecutive_same_status_count",
        ],
        "time_since_status_cols": time_since_status_cols,
        "rolling_cols": rolling_cols,
        "machine_behavior_cols": machine_behavior_cols,
        "status_behavior_cols": status_behavior_cols,
        "ratio_cols": ratio_cols,
    }

    metadata = {
        "feature_groups": feature_groups,
        "all_statuses": ALL_STATUSES,
        "normal_statuses": sorted(NORMAL_STATUSES),
        "target_statuses": sorted(TARGET_STATUSES),
        "rolling_windows_min": cfg.rolling_windows_min,
    }

    return out, metadata


df_feat, feature_metadata = build_automatic_features(df_labeled, CFG)

print("P4-10 ready")
print("Feature dataframe shape:", df_feat.shape)

print("\nFeature groups:")
for group_name, cols in feature_metadata["feature_groups"].items():
    print(f"- {group_name}: {len(cols)} columns")

display(df_feat.head())

P4-10 ready
Feature dataframe shape: (145, 132)

Feature groups:
- base_time_cols: 13 columns
- machine_sequence_cols: 2 columns
- event_gap_cols: 9 columns
- status_cols: 3 columns
- prev_status_cols: 12 columns
- last_target_cols: 3 columns
- target_history_cols: 2 columns
- status_duration_cols: 3 columns
- time_since_status_cols: 3 columns
- rolling_cols: 50 columns
- machine_behavior_cols: 6 columns
- status_behavior_cols: 4 columns
- ratio_cols: 4 columns


,process,mc_no,occurred,mc_status_raw,mc_status,mc_status_token,is_normal_status,is_target_event,is_ignored_status,next_target_time,next_target_type,y_time_sec,hour,minute,second,weekday,day,is_weekend,is_night,hour_sin,hour_cos,minute_sin,minute_cos,weekday_sin,weekday_cos,machine_event_index,time_since_first_seen_machine_sec,prev_event_time,event_gap_sec,event_gap_lag1_sec,event_gap_lag2_sec,event_gap_lag3_sec,event_gap_mean_5_sec,event_gap_std_5_sec,event_gap_median_5_sec,event_gap_min_5_sec,event_gap_max_5_sec,status_alarm_coating,status_alarm_other,status_alarm_palette,prev1_status_alarm_coating,prev1_status_alarm_other,prev1_status_alarm_palette,prev1_status_none,prev2_status_alarm_coating,prev2_status_alarm_other,prev2_status_alarm_palette,prev2_status_none,prev3_status_alarm_coating,prev3_status_alarm_other,prev3_status_alarm_palette,prev3_status_none,last_target_type_token,last_target_alarm_coating,last_target_alarm_palette,last_target_none,last_target_time,time_since_last_target_sec,target_event_count_prior,prev_status_token,status_changed,status_segment_id,status_segment_start_time,time_since_status_change_sec,consecutive_same_status_count,time_since_status_alarm_coating_sec,time_since_status_alarm_other_sec,time_since_status_alarm_palette_sec,events_5m_count,events_5m_rate_per_min,events_15m_count,events_15m_rate_per_min,events_30m_count,events_30m_rate_per_min,events_60m_count,events_60m_rate_per_min,events_120m_count,events_120m_rate_per_min,targets_5m_count,targets_5m_rate_per_min,targets_15m_count,targets_15m_rate_per_min,targets_30m_count,targets_30m_rate_per_min,targets_60m_count,targets_60m_rate_per_min,targets_120m_count,targets_120m_rate_per_min,status_alarm_coating_5m_count,status_alarm_coating_5m_rate_per_min,status_alarm_coating_15m_count,status_alarm_coating_15m_rate_per_min,status_alarm_coating_30m_count,status_alarm_coating_30m_rate_per_min,status_alarm_coating_60m_count,status_alarm_coating_60m_rate_per_min,status_alarm_coating_120m_count,status_alarm_coating_120m_rate_per_min,status_alarm_other_5m_count,status_alarm_other_5m_rate_per_min,status_alarm_other_15m_count,status_alarm_other_15m_rate_per_min,status_alarm_other_30m_count,status_alarm_other_30m_rate_per_min,status_alarm_other_60m_count,status_alarm_other_60m_rate_per_min,status_alarm_other_120m_count,status_alarm_other_120m_rate_per_min,status_alarm_palette_5m_count,status_alarm_palette_5m_rate_per_min,status_alarm_palette_15m_count,status_alarm_palette_15m_rate_per_min,status_alarm_palette_30m_count,status_alarm_palette_30m_rate_per_min,status_alarm_palette_60m_count,status_alarm_palette_60m_rate_per_min,status_alarm_palette_120m_count,status_alarm_palette_120m_rate_per_min,mc_rows,mc_target_ratio,mc_median_event_gap_sec,mc_median_y_sec,mc_p90_y_sec,mc_target_classes_seen,status_rows,status_target_ratio,status_median_y_sec,status_p90_y_sec,time_since_last_target_over_mc_median_y,event_gap_over_mc_median_event_gap,time_since_status_change_over_mc_median_y,target_event_count_prior_over_machine_events
0,apb_test,BLDC-CT-02,2026-04-03 02:13:00,Alarm Palette,alarm_palette,alarm_palette,0,1,0,2026-04-03 23:10:00,alarm_coating,75420.0,2,13,0,4,3,0,1,0.500000,0.866025,0.978148,0.207912,-0.433884,-0.900969,0,0.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,none,0,0,1,NaT,NaN,0,NaN,1,1,2026-04-03 02:13:00,0.0,0,NaN,NaN,0.0,1.0,0.2,1.0,0.066667,1.0,0.033333,1.0,0.016667,1.0,0.008333,1.0,0.2,1.0,0.066667,1.0,0.033333,1.0,0.016667,1.0,0.008333,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,1.0,0.2,1.0,0.066667,1.0,0.033333,1.0,0.016667,1.0,0.008333,116,0.586207,29700.0,58650.0,246450.0,2,40,1.0,59490.0,285930.0,NaN,NaN,0.0,0.000000
1,apb_test,BLDC-CT-02,2026-04-03 23:10:00,Alarm Coating,alarm_coating,alarm_coating,0,1,0,2026-04-05 07:20:00,alarm_coating,115800.0,23,10,0,4,3,0,0,-0.258819,0.965926,0.866025,0.500000,-0.433884,-0.900969,1,75420.0,2026-04-03 0

P4-11 — Build final feature list

In [12]:
# ============================================================
# P4-11 — Build final feature list
# ============================================================

LABEL_COLS = [
    "next_target_time",
    "next_target_type",
    "y_time_sec",
]

IDENTITY_COLS = [
    "process",
    "mc_no",
    "occurred",
    "mc_status_raw",
    "mc_status",
    "mc_status_token",
]

HELPER_COLS = [
    "is_normal_status",
    "is_target_event",
    "is_ignored_status",
    "prev_event_time",
    "prev_status_token",
    "last_target_time",
    "last_target_type_token",
    "status_segment_id",
    "status_segment_start_time",
]

EXCLUDE_FROM_FEATURES = set(LABEL_COLS + IDENTITY_COLS + HELPER_COLS)

candidate_feature_cols = []

for group_name, cols in feature_metadata["feature_groups"].items():
    for c in cols:
        if c in df_feat.columns and c not in EXCLUDE_FROM_FEATURES:
            candidate_feature_cols.append(c)

# Preserve order while removing duplicates
seen = set()
feature_cols = []
for c in candidate_feature_cols:
    if c not in seen:
        feature_cols.append(c)
        seen.add(c)

# Keep only numeric features
non_numeric_features = [
    c for c in feature_cols
    if not pd.api.types.is_numeric_dtype(df_feat[c])
]

if non_numeric_features:
    print("Dropping non-numeric feature columns:")
    print(non_numeric_features)
    feature_cols = [c for c in feature_cols if c not in non_numeric_features]

# Defensive leakage blocklist
LEAKAGE_PATTERNS = [
    "next_target",
    "y_time",
    "target_time",
]

leakage_suspect_cols = [
    c for c in feature_cols
    if any(p in c.lower() for p in LEAKAGE_PATTERNS)
]

# last_target is allowed because it means previous target type, not future next target.
leakage_suspect_cols = [
    c for c in leakage_suspect_cols
    if not c.startswith("last_target")
]

if leakage_suspect_cols:
    raise ValueError(f"Potential leakage columns found in feature_cols: {leakage_suspect_cols}")

feature_group_summary = []

for group_name, cols in feature_metadata["feature_groups"].items():
    included = [c for c in cols if c in feature_cols]
    feature_group_summary.append({
        "feature_group": group_name,
        "n_features": len(included),
        "examples": included[:8],
    })

feature_group_summary_df = pd.DataFrame(feature_group_summary)

print("P4-11 ready")
print("Total feature columns:", len(feature_cols))

print("\nFeature group summary:")
display(feature_group_summary_df)

print("\nFirst 100 feature columns:")
for i, c in enumerate(feature_cols[:100], start=1):
    print(f"{i:03d}. {c}")

if len(feature_cols) > 100:
    print(f"... {len(feature_cols) - 100} more features")

P4-11 ready
Total feature columns: 114

Feature group summary:


,feature_group,n_features,examples
0,base_time_cols,13,"[hour, minute, second, weekday, day, is_weeken..."
1,machine_sequence_cols,2,"[machine_event_index, time_since_first_seen_ma..."
2,event_gap_cols,9,"[event_gap_sec, event_gap_lag1_sec, event_gap_..."
3,status_cols,3,"[status_alarm_coating, status_alarm_other, sta..."
4,prev_status_cols,12,"[prev1_status_alarm_coating, prev1_status_alar..."
5,last_target_cols,3,"[last_target_alarm_coating, last_target_alarm_..."
6,target_history_cols,2,"[time_since_last_target_sec, target_event_coun..."
7,status_duration_cols,3,"[status_changed, time_since_status_change_sec,..."
8,time_since_status_cols,3,"[time_since_status_alarm_coating_sec, time_sin..."
9,rolling_cols,50,"[events_5m_count, events_5m_rate_per_min, even..."



First 100 feature columns:
001. hour
002. minute
003. second
004. weekday
005. day
006. is_weekend
007. is_night
008. hour_sin
009. hour_cos
010. minute_sin
011. minute_cos
012. weekday_sin
013. weekday_cos
014. machine_event_index
015. time_since_first_seen_machine_sec
016. event_gap_sec
017. event_gap_lag1_sec
018. event_gap_lag2_sec
019. event_gap_lag3_sec
020. event_gap_mean_5_sec
021. event_gap_std_5_sec
022. event_gap_median_5_sec
023. event_gap_min_5_sec
024. event_gap_max_5_sec
025. status_alarm_coating
026. status_alarm_other
027. status_alarm_palette
028. prev1_status_alarm_coating
029. prev1_status_alarm_other
030. prev1_status_alarm_palette
031. prev1_status_none
032. prev2_status_alarm_coating
033. prev2_status_alarm_other
034. prev2_status_alarm_palette
035. prev2_status_none
036. prev3_status_alarm_coating
037. prev3_status_alarm_other
038. prev3_status_alarm_palette
039. prev3_status_none
040. last_target_alarm_coating
041. last_target_alarm_palette
042. last_target_no

P4-12 — Feature validation report

In [13]:
# ============================================================
# P4-12 — Feature validation report
# ============================================================

def build_feature_validation_report(df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    checks = []

    def add(name, passed, detail, severity="ERROR"):
        checks.append({
            "check": name,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        })

    add(
        "feature_columns_exist",
        all(c in df.columns for c in feature_cols),
        f"missing_feature_cols={[c for c in feature_cols if c not in df.columns][:20]}",
    )

    add(
        "feature_count_reasonable",
        20 <= len(feature_cols) <= 500,
        f"feature_count={len(feature_cols)}",
        severity="WARNING",
    )

    non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
    add(
        "all_features_numeric",
        len(non_numeric) == 0,
        f"non_numeric_features={non_numeric[:20]}",
    )

    duplicate_cols = pd.Series(feature_cols).duplicated().sum()
    add(
        "no_duplicate_feature_names",
        duplicate_cols == 0,
        f"duplicate_feature_name_count={int(duplicate_cols)}",
    )

    leakage_bad = [
        c for c in feature_cols
        if (
            "next_target" in c.lower()
            or c.lower() == "y_time_sec"
            or c.lower().startswith("y_")
        )
    ]

    add(
        "no_obvious_future_label_leakage_features",
        len(leakage_bad) == 0,
        f"leakage_suspect_features={leakage_bad[:20]}",
    )

    all_nan_features = [c for c in feature_cols if df[c].isna().all()]
    add(
        "no_all_nan_features",
        len(all_nan_features) == 0,
        f"all_nan_features={all_nan_features[:20]}",
    )

    constant_features = []
    for c in feature_cols:
        nunique = df[c].nunique(dropna=True)
        if nunique <= 1:
            constant_features.append(c)

    add(
        "constant_features_not_too_many",
        len(constant_features) <= max(10, int(0.10 * len(feature_cols))),
        f"constant_feature_count={len(constant_features)}, examples={constant_features[:20]}",
        severity="WARNING",
    )

    inf_counts = np.isinf(df[feature_cols].select_dtypes(include=[np.number])).sum()
    inf_features = inf_counts[inf_counts > 0].index.tolist()

    add(
        "no_infinite_values",
        len(inf_features) == 0,
        f"features_with_inf={inf_features[:20]}",
    )

    return pd.DataFrame(checks)


missingness_df = pd.DataFrame({
    "feature": feature_cols,
    "missing_count": [int(df_feat[c].isna().sum()) for c in feature_cols],
    "missing_percent": [float(df_feat[c].isna().mean() * 100) for c in feature_cols],
    "nunique": [int(df_feat[c].nunique(dropna=True)) for c in feature_cols],
    "dtype": [str(df_feat[c].dtype) for c in feature_cols],
})

missingness_df = missingness_df.sort_values(
    ["missing_percent", "missing_count"],
    ascending=False
).reset_index(drop=True)

feature_validation_df = build_feature_validation_report(df_feat, feature_cols)

failed_feature_errors = feature_validation_df[
    (feature_validation_df["severity"] == "ERROR") & (~feature_validation_df["passed"])
]

READY_FOR_BATCH_4 = len(failed_feature_errors) == 0

print("=" * 100)
print("P4-12 — FEATURE VALIDATION REPORT")
print("=" * 100)

print(f"Feature dataframe rows:      {len(df_feat):,}")
print(f"Feature dataframe columns:   {df_feat.shape[1]:,}")
print(f"Final feature count:         {len(feature_cols):,}")

print("\nFeature group summary:")
display(feature_group_summary_df)

print("\nTop 30 features by missingness:")
display(missingness_df.head(30))

print("\nFeatures with >50% missing:")
display(missingness_df[missingness_df["missing_percent"] > 50].head(50))

print("\nFeature validation gates:")
display(feature_validation_df)

print("\nSample feature rows:")
sample_cols = IDENTITY_COLS + LABEL_COLS + feature_cols[:25]
sample_cols = [c for c in sample_cols if c in df_feat.columns]
display(df_feat[sample_cols].head(10))

print("=" * 100)
if READY_FOR_BATCH_4:
    print("READY_FOR_BATCH_4 = True")
    print("Batch 3 passed. Send this output to confirm before moving to Batch 4.")
else:
    print("READY_FOR_BATCH_4 = False")
    print("Batch 3 has failed ERROR gates. Fix before moving to Batch 4.")
print("=" * 100)

P4-12 — FEATURE VALIDATION REPORT
Feature dataframe rows:      145
Feature dataframe columns:   132
Final feature count:         114

Feature group summary:


,feature_group,n_features,examples
0,base_time_cols,13,"[hour, minute, second, weekday, day, is_weeken..."
1,machine_sequence_cols,2,"[machine_event_index, time_since_first_seen_ma..."
2,event_gap_cols,9,"[event_gap_sec, event_gap_lag1_sec, event_gap_..."
3,status_cols,3,"[status_alarm_coating, status_alarm_other, sta..."
4,prev_status_cols,12,"[prev1_status_alarm_coating, prev1_status_alar..."
5,last_target_cols,3,"[last_target_alarm_coating, last_target_alarm_..."
6,target_history_cols,2,"[time_since_last_target_sec, target_event_coun..."
7,status_duration_cols,3,"[status_changed, time_since_status_change_sec,..."
8,time_since_status_cols,3,"[time_since_status_alarm_coating_sec, time_sin..."
9,rolling_cols,50,"[events_5m_count, events_5m_rate_per_min, even..."



Top 30 features by missingness:


,feature,missing_count,missing_percent,nunique,dtype
0,time_since_status_alarm_palette_sec,23,15.862069,81,float64
1,event_gap_lag3_sec,8,5.517241,120,float64
2,event_gap_lag2_sec,6,4.137931,121,float64
3,event_gap_lag1_sec,4,2.758621,123,float64
4,event_gap_std_5_sec,4,2.758621,138,float64
5,time_since_status_alarm_other_sec,4,2.758621,81,float64
6,event_gap_sec,2,1.379310,125,float64
7,event_gap_mean_5_sec,2,1.379310,135,float64
8,event_gap_median_5_sec,2,1.379310,64,float64
9,event_gap_min_5_sec,2,1.379310,39,float64



Features with >50% missing:


...


Feature validation gates:


,check,severity,passed,detail
0,feature_columns_exist,ERROR,True,missing_feature_cols=[]
1,feature_count_reasonable,WARNING,True,feature_count=114
2,all_features_numeric,ERROR,True,non_numeric_features=[]
3,no_duplicate_feature_names,ERROR,True,duplicate_feature_name_count=0
4,no_obvious_future_label_leakage_features,ERROR,True,leakage_suspect_features=[]
5,no_all_nan_features,ERROR,True,all_nan_features=[]
6,constant_features_not_too_many,WARNING,True,"constant_feature_count=3, examples=['second', ..."
7,no_infinite_values,ERROR,True,features_with_inf=[]



Sample feature rows:


,process,mc_no,occurred,mc_status_raw,mc_status,mc_status_token,next_target_time,next_target_type,y_time_sec,hour,minute,second,weekday,day,is_weekend,is_night,hour_sin,hour_cos,minute_sin,minute_cos,weekday_sin,weekday_cos,machine_event_index,time_since_first_seen_machine_sec,event_gap_sec,event_gap_lag1_sec,event_gap_lag2_sec,event_gap_lag3_sec,event_gap_mean_5_sec,event_gap_std_5_sec,event_gap_median_5_sec,event_gap_min_5_sec,event_gap_max_5_sec,status_alarm_coating
0,apb_test,BLDC-CT-02,2026-04-03 02:13:00,Alarm Palette,alarm_palette,alarm_palette,2026-04-03 23:10:00,alarm_coating,75420.0,2,13,0,4,3,0,1,5.000000e-01,0.866025,9.781476e-01,2.079117e-01,-0.433884,-0.900969,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,apb_test,BLDC-CT-02,2026-04-03 23:10:00,Alarm Coating,alarm_coating,alarm_coating,2026-04-05 07:20:00,alarm_coating,115800.0,23,10,0,4,3,0,0,-2.588190e-01,0.965926,8.660254e-01,5.000000e-01,-0.433884,-0.900969,1,75420.0,75420.0,NaN,NaN,NaN,75420.0,NaN,75420.0,75420.0,75420.0,1
2,apb_test,BLDC-CT-02,2026-04-04 04:10:00,Alarm Other,alarm_other,alarm_other,2026-04-05 07:20:00,alarm_coating,97800.0,4,10,0,5,4,1,1,8.660254e-01,0.500000,8.660254e-01,5.000000e-01,-0.974928,-0.222521,2,93420.0,18000.0,75420.0,NaN,NaN,46710.0,40602.071376,46710.0,18000.0,75420.0,0
3,apb_test,BLDC-CT-02,2026-04-05 07:20:00,Alarm Coating,alarm_coating,alarm_coating,2026-04-06 15:48:00,alarm_palette,116880.0,7,20,0,6,5,1,0,9.659258e-01,-0.258819,8.660254e-01,-5.000000e-01,-0.781831,0.623490,3,191220.0,97800.0,18000.0,75420.0,NaN,63740.0,41162.201107,75420.0,18000.0,97800.0,1
4,apb_test,BLDC-CT-02,2026-04-05 13:26:00,Alarm Other,alarm_other,alarm_other,2026-04-06 15:48:00,alarm_palette,94920.0,13,26,0,6,5,1,0,-2.588190e-01,-0.965926,4.067366e-01,-9.135455e-01,-0.781831,0.623490,4,213180.0,21960.0,97800.0,18000.0,75420.0,53295.0,39572.001466,48690.0,18000.0,97800.0,0
5,apb_test,BLDC-CT-02,2026-04-06 15:48:00,Alarm Palette,alarm_palette,alarm_palette,2026-04-08 13:21:00,alarm_palette,163980.0,15,48,0,0,6,0,0,-7.071068e-01,-0.707107,-9.510565e-01,3.090170e-01,0.000000,1.000000,5,308100.0,94920.0,21960.0,97800.0,18000.0,61620.0,38999.815384,75420.0,18000.0,97800.0,0
6,apb_test,BLDC-CT-02,2026-04-07 08:50:00,Alarm Other,alarm_other,alarm_other,2026-04-08 13:21:00,alarm_palette,102660.0,8,50,0,1,7,0,0,8.660254e-01,-0.500000,-8.660254e-01,5.000000e-01,0.781831,0.623490,6,369420.0,61320.0,94920.0,21960.0,97800.0,58800.0,38255.164357,61320.0,18000.0,97800.0,0
7,apb_test,BLDC-CT-02,2026-04-07 20:30:00,Alarm Other,alarm_other,alarm_other,2026-04-08 13:21:00,alarm_palette,60660.0,20,30,0,1,7,0,0,-8.660254e-01,0.500000,5.665539e-16,-1.000000e+00,0.781831,0.623490,7,411420.0,42000.0,61320.0,94920.0,21960.0,63600.0,33000.872716,61320.0,21960.0,97800.0,0
8,apb_test,BLDC-CT-02,2026-04-08 12:15:00,Alarm Other,alarm_other,alarm_other,2026-04-08 13:21:00,alarm_palette,3960.0,12,15,0,2,8,0,0,1.224647e-16,-1.000000,1.000000e+00,2.832769e-16,0.974928,-0.222521,8,468120.0,56700.0,42000.0,61320.0,94920.0,55380.0,26908.913022,56700.0,21960.0,94920.0,0
9,apb_test,BLDC-CT-02,2026-04-08 13:21:00,Alarm Palette,alarm_palette,alarm_palette,2026-04-09 11:15:00,alarm_coating,78840.0,13,21,0,2,8,0,0,-2.588190e-01,-0.965926,8.090170e-01,-5.877853e-01,0.974928,-0.222521,9,472080.0,3960.0,56700.0,42000.0,61320.0,51780.0,33010.143895,56700.0,3960.0,94920.0,0


READY_FOR_BATCH_4 = True
Batch 3 passed. Send this output to confirm before moving to Batch 4.


P4-13 — Per-machine chronological train/val/test split

In [14]:
# ============================================================
# P4-13 — Per-machine chronological train/val/test split
# ============================================================

MIN_ROWS_FULL_SPLIT = 30

def per_machine_chrono_split(
    df: pd.DataFrame,
    cfg: AutoTrainingConfig,
    min_rows_full_split: int = 30,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Chronological split inside each process + machine.

    Machines with too few rows are kept in train only.
    This prevents tiny machines from producing meaningless val/test slices.
    """
    df_sorted = df.sort_values(["process", "mc_no", "occurred", "mc_status"]).copy()

    split_labels = pd.Series(index=df_sorted.index, dtype="object")
    reports = []

    for (process, mc_no), g in df_sorted.groupby(["process", "mc_no"], sort=False):
        g = g.sort_values(["occurred", "mc_status"])
        idx = g.index.to_numpy()
        n = len(g)

        if n < min_rows_full_split:
            train_idx = idx
            val_idx = np.array([], dtype=idx.dtype)
            test_idx = np.array([], dtype=idx.dtype)
            split_reason = "low_support_train_only"
        else:
            n_train = int(n * cfg.train_frac)
            n_val = int(n * cfg.val_frac)

            # Ensure each split has at least one row for sufficiently large machines.
            n_train = max(n_train, 1)
            n_val = max(n_val, 1)
            n_test = n - n_train - n_val

            if n_test < 1:
                n_test = 1
                n_train = max(n - n_val - n_test, 1)

            train_idx = idx[:n_train]
            val_idx = idx[n_train:n_train + n_val]
            test_idx = idx[n_train + n_val:]

            split_reason = "full_split"

        split_labels.loc[train_idx] = "train"
        split_labels.loc[val_idx] = "val"
        split_labels.loc[test_idx] = "test"

        reports.append({
            "process": process,
            "mc_no": mc_no,
            "total_rows": n,
            "train_rows": len(train_idx),
            "val_rows": len(val_idx),
            "test_rows": len(test_idx),
            "split_reason": split_reason,
            "first_time": g["occurred"].min(),
            "last_time": g["occurred"].max(),
        })

    df_sorted["_split"] = split_labels.values

    train_df = df_sorted[df_sorted["_split"].eq("train")].drop(columns=["_split"]).copy()
    val_df = df_sorted[df_sorted["_split"].eq("val")].drop(columns=["_split"]).copy()
    test_df = df_sorted[df_sorted["_split"].eq("test")].drop(columns=["_split"]).copy()

    split_report_df = pd.DataFrame(reports)

    return train_df, val_df, test_df, split_report_df


train_df, val_df, test_df, split_report_df = per_machine_chrono_split(
    df_feat,
    CFG,
    min_rows_full_split=MIN_ROWS_FULL_SPLIT,
)

split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "machines": [
        train_df["mc_no"].nunique(),
        val_df["mc_no"].nunique(),
        test_df["mc_no"].nunique(),
    ],
    "time_min": [
        train_df["occurred"].min(),
        val_df["occurred"].min(),
        test_df["occurred"].min(),
    ],
    "time_max": [
        train_df["occurred"].max(),
        val_df["occurred"].max(),
        test_df["occurred"].max(),
    ],
})

split_summary["row_percent"] = split_summary["rows"] / len(df_feat) * 100

machine_split_reason_summary = (
    split_report_df["split_reason"]
    .value_counts()
    .rename_axis("split_reason")
    .reset_index(name="machine_count")
)

print("P4-13 ready")
print(f"MIN_ROWS_FULL_SPLIT = {MIN_ROWS_FULL_SPLIT}")

print("\nSplit summary:")
display(split_summary)

print("\nMachine split reason summary:")
display(machine_split_reason_summary)

print("\nMachine split row summary:")
display(
    split_report_df[
        ["total_rows", "train_rows", "val_rows", "test_rows"]
    ].describe()
)

print("\nLowest 20 machines by total rows:")
display(split_report_df.sort_values("total_rows").head(20))

P4-13 ready
MIN_ROWS_FULL_SPLIT = 30

Split summary:


,split,rows,machines,time_min,time_max,row_percent
0,train,110,2,2026-04-03 00:05:00,2026-06-10 07:00:00,75.862069
1,val,17,1,2026-05-29 10:50:00,2026-06-08 07:00:00,11.724138
2,test,18,1,2026-06-08 17:00:00,2026-06-15 05:15:00,12.413793



Machine split reason summary:


,split_reason,machine_count
0,full_split,1
1,low_support_train_only,1



Machine split row summary:


,total_rows,train_rows,val_rows,test_rows
count,2.00000,2.000000,2.000000,2.000000
mean,72.50000,55.000000,8.500000,9.000000
std,61.51829,36.769553,12.020815,12.727922
min,29.00000,29.000000,0.000000,0.000000
25%,50.75000,42.000000,4.250000,4.500000
50%,72.50000,55.000000,8.500000,9.000000
75%,94.25000,68.000000,12.750000,13.500000
max,116.00000,81.000000,17.000000,18.000000



Lowest 20 machines by total rows:


,process,mc_no,total_rows,train_rows,val_rows,test_rows,split_reason,first_time,last_time
1,apb_test,BLDC-CT-03,29,29,0,0,low_support_train_only,2026-04-03 00:05:00,2026-06-10 07:00:00
0,apb_test,BLDC-CT-02,116,81,17,18,full_split,2026-04-03 02:13:00,2026-06-15 05:15:00


P4-14 — Train-only target clipping

In [15]:
# ============================================================
# P4-14 — Train-only target clipping
# ============================================================

clip_max = float(train_df["y_time_sec"].quantile(CFG.target_clip_quantile))

def apply_target_clip(df: pd.DataFrame, clip_max: float) -> pd.DataFrame:
    out = df.copy()
    out["y_time_sec_clip"] = out["y_time_sec"].clip(lower=0, upper=clip_max)
    out["y_time_was_clipped"] = (out["y_time_sec"] > clip_max).astype(np.int8)
    return out


train_df = apply_target_clip(train_df, clip_max)
val_df = apply_target_clip(val_df, clip_max)
test_df = apply_target_clip(test_df, clip_max)

target_clip_report = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "clip_max_sec": [clip_max, clip_max, clip_max],
    "clip_max_readable": [seconds_to_readable(clip_max)] * 3,
    "clipped_rows": [
        int(train_df["y_time_was_clipped"].sum()),
        int(val_df["y_time_was_clipped"].sum()),
        int(test_df["y_time_was_clipped"].sum()),
    ],
    "clipped_percent": [
        train_df["y_time_was_clipped"].mean() * 100,
        val_df["y_time_was_clipped"].mean() * 100,
        test_df["y_time_was_clipped"].mean() * 100,
    ],
    "raw_median_sec": [
        train_df["y_time_sec"].median(),
        val_df["y_time_sec"].median(),
        test_df["y_time_sec"].median(),
    ],
    "raw_p90_sec": [
        train_df["y_time_sec"].quantile(0.90),
        val_df["y_time_sec"].quantile(0.90),
        test_df["y_time_sec"].quantile(0.90),
    ],
    "raw_p99_sec": [
        train_df["y_time_sec"].quantile(0.99),
        val_df["y_time_sec"].quantile(0.99),
        test_df["y_time_sec"].quantile(0.99),
    ],
    "raw_max_sec": [
        train_df["y_time_sec"].max(),
        val_df["y_time_sec"].max(),
        test_df["y_time_sec"].max(),
    ],
})

print("P4-14 ready")
print(f"Train-only clip quantile: {CFG.target_clip_quantile}")
print(f"clip_max = {clip_max:.3f} sec ({seconds_to_readable(clip_max)})")

display(target_clip_report)

P4-14 ready
Train-only clip quantile: 0.99
clip_max = 841371.000 sec (9.74d)


,split,rows,clip_max_sec,clip_max_readable,clipped_rows,clipped_percent,raw_median_sec,raw_p90_sec,raw_p99_sec,raw_max_sec
0,train,110,841371.0,9.74d,2,1.818182,87900.0,462330.0,841371.0,864000.0
1,val,17,841371.0,9.74d,0,0.000000,34800.0,161820.0,360720.0,397200.0
2,test,18,841371.0,9.74d,0,0.000000,44700.0,92370.0,154728.0,163500.0


P4-15 — Overwrite behavior features using train-only maps

This is the leakage-safety fix.

In [16]:
# ============================================================
# P4-15 — Overwrite behavior features using train-only maps
# ============================================================

MACHINE_BEHAVIOR_COLS = feature_metadata["feature_groups"]["machine_behavior_cols"]
STATUS_BEHAVIOR_COLS = feature_metadata["feature_groups"]["status_behavior_cols"]
RATIO_COLS = feature_metadata["feature_groups"]["ratio_cols"]

def fit_train_only_behavior_maps(train: pd.DataFrame) -> dict:
    """
    Fit machine/status behavior maps using train split only.
    This prevents validation/test leakage.
    """
    machine_behavior_train = (
        train.groupby(["process", "mc_no"])
        .agg(
            mc_rows=("mc_no", "size"),
            mc_target_ratio=("is_target_event", "mean"),
            mc_median_event_gap_sec=("event_gap_sec", "median"),
            mc_median_y_sec=("y_time_sec_clip", "median"),
            mc_p90_y_sec=("y_time_sec_clip", lambda x: x.quantile(0.90)),
            mc_target_classes_seen=("next_target_type", "nunique"),
        )
        .reset_index()
    )

    status_behavior_train = (
        train.groupby("mc_status")
        .agg(
            status_rows=("mc_status", "size"),
            status_target_ratio=("is_target_event", "mean"),
            status_median_y_sec=("y_time_sec_clip", "median"),
            status_p90_y_sec=("y_time_sec_clip", lambda x: x.quantile(0.90)),
        )
        .reset_index()
    )

    global_machine_behavior = {
        "mc_rows": float(train.groupby(["process", "mc_no"]).size().median()),
        "mc_target_ratio": float(train["is_target_event"].mean()),
        "mc_median_event_gap_sec": float(train["event_gap_sec"].median()),
        "mc_median_y_sec": float(train["y_time_sec_clip"].median()),
        "mc_p90_y_sec": float(train["y_time_sec_clip"].quantile(0.90)),
        "mc_target_classes_seen": float(train["next_target_type"].nunique()),
    }

    global_status_behavior = {
        "status_rows": float(train["mc_status"].value_counts().median()),
        "status_target_ratio": float(train["is_target_event"].mean()),
        "status_median_y_sec": float(train["y_time_sec_clip"].median()),
        "status_p90_y_sec": float(train["y_time_sec_clip"].quantile(0.90)),
    }

    return {
        "machine_behavior_train": machine_behavior_train,
        "status_behavior_train": status_behavior_train,
        "global_machine_behavior": global_machine_behavior,
        "global_status_behavior": global_status_behavior,
    }


def apply_train_only_behavior_maps(df: pd.DataFrame, maps: dict) -> pd.DataFrame:
    out = df.copy()

    # Remove previously full-data-derived behavior columns before re-merging.
    out = out.drop(
        columns=[c for c in MACHINE_BEHAVIOR_COLS + STATUS_BEHAVIOR_COLS if c in out.columns],
        errors="ignore",
    )

    machine_map = maps["machine_behavior_train"]
    status_map = maps["status_behavior_train"]

    out = out.merge(machine_map, on=["process", "mc_no"], how="left")
    out = out.merge(status_map, on="mc_status", how="left")

    for c, v in maps["global_machine_behavior"].items():
        if c in out.columns:
            out[c] = out[c].fillna(v)

    for c, v in maps["global_status_behavior"].items():
        if c in out.columns:
            out[c] = out[c].fillna(v)

    # Recompute ratio features using train-only behavior maps.
    eps = 1.0

    out["time_since_last_target_over_mc_median_y"] = (
        out["time_since_last_target_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["event_gap_over_mc_median_event_gap"] = (
        out["event_gap_sec"] / (out["mc_median_event_gap_sec"] + eps)
    )

    out["time_since_status_change_over_mc_median_y"] = (
        out["time_since_status_change_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["target_event_count_prior_over_machine_events"] = (
        out["target_event_count_prior"] / (out["machine_event_index"] + 1)
    )

    return out


behavior_maps = fit_train_only_behavior_maps(train_df)

train_df = apply_train_only_behavior_maps(train_df, behavior_maps)
val_df = apply_train_only_behavior_maps(val_df, behavior_maps)
test_df = apply_train_only_behavior_maps(test_df, behavior_maps)

behavior_map_report = {
    "machine_behavior_map_rows": int(len(behavior_maps["machine_behavior_train"])),
    "status_behavior_map_rows": int(len(behavior_maps["status_behavior_train"])),
    "global_machine_behavior": behavior_maps["global_machine_behavior"],
    "global_status_behavior": behavior_maps["global_status_behavior"],
}

print("P4-15 ready")
print(json.dumps(behavior_map_report, indent=2, ensure_ascii=False))

print("\nTrain-only machine behavior map sample:")
display(behavior_maps["machine_behavior_train"].head(10))

print("\nTrain-only status behavior map:")
display(behavior_maps["status_behavior_train"])

P4-15 ready
{
  "machine_behavior_map_rows": 2,
  "status_behavior_map_rows": 3,
  "global_machine_behavior": {
    "mc_rows": 55.0,
    "mc_target_ratio": 0.5727272727272728,
    "mc_median_event_gap_sec": 47400.0,
    "mc_median_y_sec": 87900.0,
    "mc_p90_y_sec": 462330.0,
    "mc_target_classes_seen": 2.0
  },
  "global_status_behavior": {
    "status_rows": 34.0,
    "status_target_ratio": 0.5727272727272728,
    "status_median_y_sec": 87900.0,
    "status_p90_y_sec": 462330.0
  }
}

Train-only machine behavior map sample:


,process,mc_no,mc_rows,mc_target_ratio,mc_median_event_gap_sec,mc_median_y_sec,mc_p90_y_sec,mc_target_classes_seen
0,apb_test,BLDC-CT-02,81,0.567901,32550.0,60660.0,288600.0,2
1,apb_test,BLDC-CT-03,29,0.586207,165000.0,349500.0,606960.0,2



Train-only status behavior map:


,mc_status,status_rows,status_target_ratio,status_median_y_sec,status_p90_y_sec
0,alarm_coating,34,1.0,101250.0,515160.0
1,alarm_other,47,0.0,102120.0,367680.0
2,alarm_palette,29,1.0,71820.0,445356.0


P4-16 — Train-only feature fill values

In [17]:
# ============================================================
# P4-16 — Train-only feature fill values
# ============================================================

def build_feature_fill_values(train: pd.DataFrame, feature_cols: list) -> dict:
    """
    Build train-only fill values.

    Strategy:
      - time_since_status_*: use train p99 as "not seen recently / unseen yet" sentinel
      - time_since_last_target_sec: use train p99
      - event-gap columns: use median
      - all others: use median
      - if no valid value exists: use 0.0
    """
    fill_values = {}

    train_numeric = train[feature_cols].replace([np.inf, -np.inf], np.nan)

    for c in feature_cols:
        s = train_numeric[c]

        if c.startswith("time_since_status_") or c == "time_since_last_target_sec":
            val = s.quantile(0.99)
        else:
            val = s.median()

        if pd.isna(val) or np.isinf(val):
            val = 0.0

        fill_values[c] = float(val)

    return fill_values


feature_fill_values = build_feature_fill_values(train_df, feature_cols)

feature_fill_df = pd.DataFrame({
    "feature": list(feature_fill_values.keys()),
    "fill_value": list(feature_fill_values.values()),
})

feature_fill_df["fill_readable_if_sec"] = np.where(
    feature_fill_df["feature"].str.endswith("_sec"),
    feature_fill_df["fill_value"].apply(seconds_to_readable),
    "",
)

print("P4-16 ready")
print(f"Feature fill values fitted from train only: {len(feature_fill_values):,}")

print("\nTop 30 largest fill values:")
display(feature_fill_df.sort_values("fill_value", ascending=False).head(30))

print("\nFill values for time_since_status features:")
display(feature_fill_df[feature_fill_df["feature"].str.startswith("time_since_status_")])

P4-16 ready
Feature fill values fitted from train only: 114

Top 30 largest fill values:


,feature,fill_value,fill_readable_if_sec
14,time_since_first_seen_machine_sec,2.230320e+06,25.81d
48,time_since_status_alarm_other_sec,1.180035e+06,13.66d
45,time_since_status_change_sec,1.005825e+06,11.64d
42,time_since_last_target_sec,8.577780e+05,9.93d
47,time_since_status_alarm_coating_sec,7.723320e+05,8.94d
49,time_since_status_alarm_palette_sec,5.558988e+05,6.43d
109,status_p90_y_sec,4.453560e+05,5.15d
104,mc_p90_y_sec,2.886000e+05,3.34d
23,event_gap_max_5_sec,1.726200e+05,2.00d
108,status_median_y_sec,1.012500e+05,1.17d



Fill values for time_since_status features:


,feature,fill_value,fill_readable_if_sec
45,time_since_status_change_sec,1.005825e+06,11.64d
47,time_since_status_alarm_coating_sec,7.723320e+05,8.94d
48,time_since_status_alarm_other_sec,1.180035e+06,13.66d
49,time_since_status_alarm_palette_sec,5.558988e+05,6.43d
112,time_since_status_change_over_mc_median_y,5.556734e+00,


P4-17 — Machine-balanced sample weights and final train matrices

In [18]:
# ============================================================
# P4-17 — Machine-balanced sample weights and final train matrices
# ============================================================

def build_model_matrix(df: pd.DataFrame, feature_cols: list, fill_values: dict) -> pd.DataFrame:
    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(fill_values)

    # Final safety check
    bad_cols = X.columns[X.isna().any()].tolist()
    if bad_cols:
        raise ValueError(f"NaN still found after filling. Example columns: {bad_cols[:20]}")

    inf_counts = np.isinf(X.select_dtypes(include=[np.number])).sum()
    inf_cols = inf_counts[inf_counts > 0].index.tolist()
    if inf_cols:
        raise ValueError(f"Inf still found after filling. Example columns: {inf_cols[:20]}")

    return X.astype(np.float32)


def make_machine_balanced_weights(df: pd.DataFrame) -> np.ndarray:
    """
    Down-weight high-volume machines, up-weight low-volume machines.
    Normalized to mean 1.0.
    """
    counts = df.groupby(["process", "mc_no"]).size()

    keys = list(zip(df["process"], df["mc_no"]))
    raw_w = np.array([1.0 / math.sqrt(counts.loc[k]) for k in keys], dtype=np.float64)

    raw_w = raw_w / raw_w.mean()
    return raw_w.astype(np.float32)


X_train = build_model_matrix(train_df, feature_cols, feature_fill_values)
X_val = build_model_matrix(val_df, feature_cols, feature_fill_values)
X_test = build_model_matrix(test_df, feature_cols, feature_fill_values)

y_train_raw = train_df["y_time_sec"].to_numpy(dtype=np.float64)
y_val_raw = val_df["y_time_sec"].to_numpy(dtype=np.float64)
y_test_raw = test_df["y_time_sec"].to_numpy(dtype=np.float64)

y_train_clip = train_df["y_time_sec_clip"].to_numpy(dtype=np.float64)
y_val_clip = val_df["y_time_sec_clip"].to_numpy(dtype=np.float64)
y_test_clip = test_df["y_time_sec_clip"].to_numpy(dtype=np.float64)

y_type_train = train_df["next_target_type"].astype(str).to_numpy()
y_type_val = val_df["next_target_type"].astype(str).to_numpy()
y_type_test = test_df["next_target_type"].astype(str).to_numpy()

sample_weight_train_eta = make_machine_balanced_weights(train_df)

matrix_report = pd.DataFrame({
    "matrix": [
        "X_train", "X_val", "X_test",
        "y_train_raw", "y_val_raw", "y_test_raw",
        "y_train_clip", "y_val_clip", "y_test_clip",
        "y_type_train", "y_type_val", "y_type_test",
        "sample_weight_train_eta",
    ],
    "shape_or_length": [
        str(X_train.shape), str(X_val.shape), str(X_test.shape),
        len(y_train_raw), len(y_val_raw), len(y_test_raw),
        len(y_train_clip), len(y_val_clip), len(y_test_clip),
        len(y_type_train), len(y_type_val), len(y_type_test),
        len(sample_weight_train_eta),
    ],
})

weight_summary = pd.Series(sample_weight_train_eta).describe().to_frame("sample_weight_train_eta")

split_target_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "machines": [
        train_df["mc_no"].nunique(),
        val_df["mc_no"].nunique(),
        test_df["mc_no"].nunique(),
    ],
    "target_median_sec": [
        train_df["y_time_sec"].median(),
        val_df["y_time_sec"].median(),
        test_df["y_time_sec"].median(),
    ],
    "target_p90_sec": [
        train_df["y_time_sec"].quantile(0.90),
        val_df["y_time_sec"].quantile(0.90),
        test_df["y_time_sec"].quantile(0.90),
    ],
    "target_p99_sec": [
        train_df["y_time_sec"].quantile(0.99),
        val_df["y_time_sec"].quantile(0.99),
        test_df["y_time_sec"].quantile(0.99),
    ],
    "target_max_sec": [
        train_df["y_time_sec"].max(),
        val_df["y_time_sec"].max(),
        test_df["y_time_sec"].max(),
    ],
})

# Check feature-column consistency
same_columns_train_val = list(X_train.columns) == list(X_val.columns)
same_columns_train_test = list(X_train.columns) == list(X_test.columns)

final_checks = []

def add_final_check(name, passed, detail, severity="ERROR"):
    final_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })

add_final_check(
    "x_matrices_have_same_columns",
    same_columns_train_val and same_columns_train_test,
    f"train_val_same={same_columns_train_val}, train_test_same={same_columns_train_test}",
)

add_final_check(
    "x_train_rows_match_y_train",
    len(X_train) == len(y_train_clip) == len(y_type_train) == len(sample_weight_train_eta),
    f"X_train={len(X_train)}, y_train_clip={len(y_train_clip)}, y_type_train={len(y_type_train)}, weights={len(sample_weight_train_eta)}",
)

add_final_check(
    "x_val_rows_match_y_val",
    len(X_val) == len(y_val_clip) == len(y_type_val),
    f"X_val={len(X_val)}, y_val_clip={len(y_val_clip)}, y_type_val={len(y_type_val)}",
)

add_final_check(
    "x_test_rows_match_y_test",
    len(X_test) == len(y_test_clip) == len(y_type_test),
    f"X_test={len(X_test)}, y_test_clip={len(y_test_clip)}, y_type_test={len(y_type_test)}",
)

add_final_check(
    "no_nan_in_model_matrices",
    not X_train.isna().any().any() and not X_val.isna().any().any() and not X_test.isna().any().any(),
    "NaN check passed for X_train/X_val/X_test",
)

add_final_check(
    "feature_count_matches",
    X_train.shape[1] == len(feature_cols),
    f"X_train_features={X_train.shape[1]}, feature_cols={len(feature_cols)}",
)

add_final_check(
    "clip_max_is_positive",
    clip_max > 0,
    f"clip_max={clip_max:.3f}",
)

add_final_check(
    "train_val_test_non_empty",
    len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0,
    f"train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}",
)

final_matrix_validation_df = pd.DataFrame(final_checks)

failed_matrix_errors = final_matrix_validation_df[
    (final_matrix_validation_df["severity"] == "ERROR") & (~final_matrix_validation_df["passed"])
]

READY_FOR_BATCH_5 = len(failed_matrix_errors) == 0

print("=" * 100)
print("P4-17 — FINAL TRAIN/VAL/TEST MATRIX REPORT")
print("=" * 100)

print(f"Final feature count: {len(feature_cols):,}")
print(f"clip_max: {clip_max:.3f} sec ({seconds_to_readable(clip_max)})")

print("\nSplit summary:")
display(split_summary)

print("\nMachine split reason summary:")
display(machine_split_reason_summary)

print("\nTarget clipping report:")
display(target_clip_report)

print("\nSplit target summary:")
display(split_target_summary)

print("\nMatrix report:")
display(matrix_report)

print("\nSample weight summary:")
display(weight_summary)

print("\nBehavior map report:")
print(json.dumps(behavior_map_report, indent=2, ensure_ascii=False))

print("\nFinal matrix validation gates:")
display(final_matrix_validation_df)

print("=" * 100)
if READY_FOR_BATCH_5:
    print("READY_FOR_BATCH_5 = True")
    print("Batch 4 passed. Send this output to confirm before moving to Batch 5.")
else:
    print("READY_FOR_BATCH_5 = False")
    print("Batch 4 has failed ERROR gates. Fix before moving to Batch 5.")
print("=" * 100)

P4-17 — FINAL TRAIN/VAL/TEST MATRIX REPORT
Final feature count: 114
clip_max: 841371.000 sec (9.74d)

Split summary:


,split,rows,machines,time_min,time_max,row_percent
0,train,110,2,2026-04-03 00:05:00,2026-06-10 07:00:00,75.862069
1,val,17,1,2026-05-29 10:50:00,2026-06-08 07:00:00,11.724138
2,test,18,1,2026-06-08 17:00:00,2026-06-15 05:15:00,12.413793



Machine split reason summary:


,split_reason,machine_count
0,full_split,1
1,low_support_train_only,1



Target clipping report:


...


Sample weight summary:


,sample_weight_train_eta
count,110.000000
mean,1.000000
std,0.252439
min,0.849641
25%,0.849641
50%,0.849641
75%,1.419969
max,1.419969



Behavior map report:
{
  "machine_behavior_map_rows": 2,
  "status_behavior_map_rows": 3,
  "global_machine_behavior": {
    "mc_rows": 55.0,
    "mc_target_ratio": 0.5727272727272728,
    "mc_median_event_gap_sec": 47400.0,
    "mc_median_y_sec": 87900.0,
    "mc_p90_y_sec": 462330.0,
    "mc_target_classes_seen": 2.0
  },
  "global_status_behavior": {
    "status_rows": 34.0,
    "status_target_ratio": 0.5727272727272728,
    "status_median_y_sec": 87900.0,
    "status_p90_y_sec": 462330.0
  }
}

Final matrix validation gates:


,check,severity,passed,detail
0,x_matrices_have_same_columns,ERROR,True,"train_val_same=True, train_test_same=True"
1,x_train_rows_match_y_train,ERROR,True,"X_train=110, y_train_clip=110, y_type_train=11..."
2,x_val_rows_match_y_val,ERROR,True,"X_val=17, y_val_clip=17, y_type_val=17"
3,x_test_rows_match_y_test,ERROR,True,"X_test=18, y_test_clip=18, y_type_test=18"
4,no_nan_in_model_matrices,ERROR,True,NaN check passed for X_train/X_val/X_test
5,feature_count_matches,ERROR,True,"X_train_features=114, feature_cols=114"
6,clip_max_is_positive,ERROR,True,clip_max=841371.000
7,train_val_test_non_empty,ERROR,True,"train=110, val=17, test=18"


READY_FOR_BATCH_5 = True
Batch 4 passed. Send this output to confirm before moving to Batch 5.


P4-17.1 — Cap machine-balanced sample weights

In [19]:
# ============================================================
# P4-17.1 — Cap machine-balanced sample weights
# ============================================================

SAMPLE_WEIGHT_CAP = 5.0

sample_weight_train_eta_uncapped = sample_weight_train_eta.copy()

sample_weight_train_eta = np.minimum(
    sample_weight_train_eta,
    SAMPLE_WEIGHT_CAP
).astype(np.float32)

# Renormalize to mean 1.0
sample_weight_train_eta = (
    sample_weight_train_eta / sample_weight_train_eta.mean()
).astype(np.float32)

weight_cap_report = pd.DataFrame({
    "metric": [
        "cap_value",
        "uncapped_min",
        "uncapped_mean",
        "uncapped_p50",
        "uncapped_p95",
        "uncapped_p99",
        "uncapped_max",
        "capped_min",
        "capped_mean",
        "capped_p50",
        "capped_p95",
        "capped_p99",
        "capped_max",
        "rows_affected_by_cap",
        "rows_affected_by_cap_percent",
    ],
    "value": [
        SAMPLE_WEIGHT_CAP,
        float(np.min(sample_weight_train_eta_uncapped)),
        float(np.mean(sample_weight_train_eta_uncapped)),
        float(np.quantile(sample_weight_train_eta_uncapped, 0.50)),
        float(np.quantile(sample_weight_train_eta_uncapped, 0.95)),
        float(np.quantile(sample_weight_train_eta_uncapped, 0.99)),
        float(np.max(sample_weight_train_eta_uncapped)),
        float(np.min(sample_weight_train_eta)),
        float(np.mean(sample_weight_train_eta)),
        float(np.quantile(sample_weight_train_eta, 0.50)),
        float(np.quantile(sample_weight_train_eta, 0.95)),
        float(np.quantile(sample_weight_train_eta, 0.99)),
        float(np.max(sample_weight_train_eta)),
        int((sample_weight_train_eta_uncapped > SAMPLE_WEIGHT_CAP).sum()),
        float((sample_weight_train_eta_uncapped > SAMPLE_WEIGHT_CAP).mean() * 100),
    ],
})

print("P4-17.1 ready")
display(weight_cap_report)

P4-17.1 ready


,metric,value
0,cap_value,5.000000
1,uncapped_min,0.849641
2,uncapped_mean,1.000000
3,uncapped_p50,0.849641
4,uncapped_p95,1.419969
5,uncapped_p99,1.419969
6,uncapped_max,1.419969
7,capped_min,0.849641
8,capped_mean,1.000000
9,capped_p50,0.849641


P4-18 — Build ETA and next-type baselines

In [20]:
# ============================================================
# P4-18 — Build ETA and next-type baselines
# ============================================================

def mode_or_none(x):
    m = x.mode(dropna=True)
    if len(m) == 0:
        return None
    return str(m.iloc[0])


def fit_eta_baselines(train: pd.DataFrame) -> dict:
    """
    Fit simple train-only ETA baselines.

    Uses clipped target for robust baseline fitting.
    Evaluated later against raw y_time_sec.
    """
    eta_maps = {}

    eta_maps["global_median"] = float(train["y_time_sec_clip"].median())

    eta_maps["status_median"] = (
        train.groupby("mc_status")["y_time_sec_clip"]
        .median()
        .astype(float)
        .to_dict()
    )

    eta_maps["machine_median"] = (
        train.groupby(["process", "mc_no"])["y_time_sec_clip"]
        .median()
        .astype(float)
        .to_dict()
    )

    eta_maps["machine_status_median"] = (
        train.groupby(["process", "mc_no", "mc_status"])["y_time_sec_clip"]
        .median()
        .astype(float)
        .to_dict()
    )

    return eta_maps


def predict_eta_baselines(df: pd.DataFrame, eta_maps: dict) -> pd.DataFrame:
    """
    Predict ETA using multiple baseline strategies.
    """
    global_median = eta_maps["global_median"]

    status_map = eta_maps["status_median"]
    machine_map = eta_maps["machine_median"]
    machine_status_map = eta_maps["machine_status_median"]

    pred_global = np.full(len(df), global_median, dtype=np.float64)

    pred_status = np.array([
        status_map.get(status, global_median)
        for status in df["mc_status"]
    ], dtype=np.float64)

    pred_machine = np.array([
        machine_map.get((process, mc_no), global_median)
        for process, mc_no in zip(df["process"], df["mc_no"])
    ], dtype=np.float64)

    pred_machine_status = np.array([
        machine_status_map.get(
            (process, mc_no, status),
            machine_map.get((process, mc_no), status_map.get(status, global_median))
        )
        for process, mc_no, status in zip(df["process"], df["mc_no"], df["mc_status"])
    ], dtype=np.float64)

    return pd.DataFrame({
        "global_median_eta": pred_global,
        "status_median_eta": pred_status,
        "machine_median_eta": pred_machine,
        "machine_status_median_eta": pred_machine_status,
    }, index=df.index)


def fit_type_baselines(train: pd.DataFrame) -> dict:
    """
    Fit train-only next-target-type baselines.
    """
    type_maps = {}

    global_mode = mode_or_none(train["next_target_type"])
    type_maps["global_mode"] = global_mode

    type_maps["status_mode"] = (
        train.groupby("mc_status")["next_target_type"]
        .agg(mode_or_none)
        .to_dict()
    )

    type_maps["machine_mode"] = (
        train.groupby(["process", "mc_no"])["next_target_type"]
        .agg(mode_or_none)
        .to_dict()
    )

    type_maps["machine_status_mode"] = (
        train.groupby(["process", "mc_no", "mc_status"])["next_target_type"]
        .agg(mode_or_none)
        .to_dict()
    )

    return type_maps


def predict_type_baselines(df: pd.DataFrame, type_maps: dict) -> pd.DataFrame:
    global_mode = type_maps["global_mode"]

    status_map = type_maps["status_mode"]
    machine_map = type_maps["machine_mode"]
    machine_status_map = type_maps["machine_status_mode"]

    pred_global = np.array([global_mode] * len(df), dtype=object)

    pred_status = np.array([
        status_map.get(status, global_mode)
        for status in df["mc_status"]
    ], dtype=object)

    pred_machine = np.array([
        machine_map.get((process, mc_no), global_mode)
        for process, mc_no in zip(df["process"], df["mc_no"])
    ], dtype=object)

    pred_machine_status = np.array([
        machine_status_map.get(
            (process, mc_no, status),
            machine_map.get((process, mc_no), status_map.get(status, global_mode))
        )
        for process, mc_no, status in zip(df["process"], df["mc_no"], df["mc_status"])
    ], dtype=object)

    return pd.DataFrame({
        "global_mode_type": pred_global,
        "status_mode_type": pred_status,
        "machine_mode_type": pred_machine,
        "machine_status_mode_type": pred_machine_status,
    }, index=df.index)


eta_baseline_maps = fit_eta_baselines(train_df)
type_baseline_maps = fit_type_baselines(train_df)

eta_baseline_pred_val = predict_eta_baselines(val_df, eta_baseline_maps)
eta_baseline_pred_test = predict_eta_baselines(test_df, eta_baseline_maps)

type_baseline_pred_val = predict_type_baselines(val_df, type_baseline_maps)
type_baseline_pred_test = predict_type_baselines(test_df, type_baseline_maps)

print("P4-18 ready")

print("\nETA baseline global median:")
print(f"{eta_baseline_maps['global_median']:.3f} sec ({seconds_to_readable(eta_baseline_maps['global_median'])})")

print("\nType baseline global mode:")
print(type_baseline_maps["global_mode"])

print("\nETA baseline predictions sample:")
display(eta_baseline_pred_val.head())

print("\nType baseline predictions sample:")
display(type_baseline_pred_val.head())

P4-18 ready

ETA baseline global median:
87900.000 sec (1.02d)

Type baseline global mode:
alarm_palette

ETA baseline predictions sample:


,global_median_eta,status_median_eta,machine_median_eta,machine_status_median_eta
0,87900.0,71820.0,60660.0,69300.0
1,87900.0,101250.0,60660.0,55800.0
2,87900.0,102120.0,60660.0,60660.0
3,87900.0,102120.0,60660.0,60660.0
4,87900.0,101250.0,60660.0,55800.0



Type baseline predictions sample:


,global_mode_type,status_mode_type,machine_mode_type,machine_status_mode_type
0,alarm_palette,alarm_palette,alarm_palette,alarm_palette
1,alarm_palette,alarm_coating,alarm_palette,alarm_palette
2,alarm_palette,alarm_palette,alarm_palette,alarm_palette
3,alarm_palette,alarm_palette,alarm_palette,alarm_palette
4,alarm_palette,alarm_coating,alarm_palette,alarm_palette


P4-19 — Baseline evaluation report

In [21]:
# ============================================================
# P4-19 — Baseline evaluation report
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, eps)) * 100


def hit_rate(y_true, y_pred, tolerance_sec):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return np.mean(np.abs(y_true - y_pred) <= tolerance_sec) * 100


def evaluate_eta_predictions(y_true_raw, y_pred, name, split):
    y_true_raw = np.asarray(y_true_raw, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    rmse = math.sqrt(mean_squared_error(y_true_raw, y_pred))

    return {
        "split": split,
        "baseline": name,
        "MAE_sec": mean_absolute_error(y_true_raw, y_pred),
        "RMSE_sec": rmse,
        "MedAE_sec": median_absolute_error(y_true_raw, y_pred),
        "SMAPE_percent": smape(y_true_raw, y_pred),
        "Hit_2m_percent": hit_rate(y_true_raw, y_pred, 120),
        "Hit_5m_percent": hit_rate(y_true_raw, y_pred, 300),
        "Hit_10m_percent": hit_rate(y_true_raw, y_pred, 600),
        "pred_median_sec": float(np.median(y_pred)),
        "pred_p90_sec": float(np.quantile(y_pred, 0.90)),
        "actual_median_sec": float(np.median(y_true_raw)),
        "actual_p90_sec": float(np.quantile(y_true_raw, 0.90)),
    }


def evaluate_type_predictions(y_true, y_pred, name, split):
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)

    return {
        "split": split,
        "baseline": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


eta_baseline_rows = []

for split_name, df_split, y_true_raw, pred_df in [
    ("val", val_df, y_val_raw, eta_baseline_pred_val),
    ("test_reference", test_df, y_test_raw, eta_baseline_pred_test),
]:
    for col in pred_df.columns:
        eta_baseline_rows.append(
            evaluate_eta_predictions(
                y_true_raw=y_true_raw,
                y_pred=pred_df[col].to_numpy(),
                name=col,
                split=split_name,
            )
        )

eta_baseline_report = pd.DataFrame(eta_baseline_rows)

eta_baseline_report_sorted = eta_baseline_report.sort_values(
    ["split", "MedAE_sec", "Hit_5m_percent"],
    ascending=[True, True, False],
).reset_index(drop=True)

type_baseline_rows = []

for split_name, y_true_type, pred_df in [
    ("val", y_type_val, type_baseline_pred_val),
    ("test_reference", y_type_test, type_baseline_pred_test),
]:
    for col in pred_df.columns:
        type_baseline_rows.append(
            evaluate_type_predictions(
                y_true=y_true_type,
                y_pred=pred_df[col].to_numpy(),
                name=col,
                split=split_name,
            )
        )

type_baseline_report = pd.DataFrame(type_baseline_rows)

type_baseline_report_sorted = type_baseline_report.sort_values(
    ["split", "weighted_f1", "accuracy"],
    ascending=[True, False, False],
).reset_index(drop=True)

# Best validation baselines
best_eta_baseline_name = (
    eta_baseline_report[eta_baseline_report["split"].eq("val")]
    .sort_values(["MedAE_sec", "Hit_5m_percent"], ascending=[True, False])
    .iloc[0]["baseline"]
)

best_type_baseline_name = (
    type_baseline_report[type_baseline_report["split"].eq("val")]
    .sort_values(["weighted_f1", "accuracy"], ascending=[False, False])
    .iloc[0]["baseline"]
)

best_eta_baseline_val = eta_baseline_report[
    (eta_baseline_report["split"].eq("val")) &
    (eta_baseline_report["baseline"].eq(best_eta_baseline_name))
].iloc[0].to_dict()

best_type_baseline_val = type_baseline_report[
    (type_baseline_report["split"].eq("val")) &
    (type_baseline_report["baseline"].eq(best_type_baseline_name))
].iloc[0].to_dict()

# Detailed classification report for best validation type baseline
best_type_val_pred = type_baseline_pred_val[best_type_baseline_name].astype(str).to_numpy()

type_classes_sorted = sorted(set(y_type_val) | set(best_type_val_pred))

best_type_classification_report_dict = classification_report(
    y_type_val,
    best_type_val_pred,
    labels=type_classes_sorted,
    output_dict=True,
    zero_division=0,
)

best_type_classification_report_df = pd.DataFrame(
    best_type_classification_report_dict
).T

best_type_confusion = pd.DataFrame(
    confusion_matrix(y_type_val, best_type_val_pred, labels=type_classes_sorted),
    index=[f"true_{c}" for c in type_classes_sorted],
    columns=[f"pred_{c}" for c in type_classes_sorted],
)

baseline_summary = {
    "best_eta_baseline_val": best_eta_baseline_name,
    "best_eta_baseline_val_MedAE_sec": float(best_eta_baseline_val["MedAE_sec"]),
    "best_eta_baseline_val_Hit_5m_percent": float(best_eta_baseline_val["Hit_5m_percent"]),
    "best_eta_baseline_val_Hit_10m_percent": float(best_eta_baseline_val["Hit_10m_percent"]),
    "best_type_baseline_val": best_type_baseline_name,
    "best_type_baseline_val_accuracy": float(best_type_baseline_val["accuracy"]),
    "best_type_baseline_val_macro_f1": float(best_type_baseline_val["macro_f1"]),
    "best_type_baseline_val_weighted_f1": float(best_type_baseline_val["weighted_f1"]),
}

print("=" * 100)
print("P4-19 — BASELINE EVALUATION REPORT")
print("=" * 100)

print("\nBaseline summary:")
print(json.dumps(baseline_summary, indent=2, ensure_ascii=False))

print("\nETA baseline report:")
display(eta_baseline_report_sorted)

print("\nNext-type baseline report:")
display(type_baseline_report_sorted)

print(f"\nBest validation next-type baseline: {best_type_baseline_name}")
print("\nClassification report for best validation next-type baseline:")
display(best_type_classification_report_df)

print("\nConfusion matrix for best validation next-type baseline:")
display(best_type_confusion)

print("=" * 100)
print("READY_FOR_BATCH_6 = True")
print("Batch 5 complete. Send this output before moving to Batch 6.")
print("=" * 100)

P4-19 — BASELINE EVALUATION REPORT

Baseline summary:
{
  "best_eta_baseline_val": "machine_status_median_eta",
  "best_eta_baseline_val_MedAE_sec": 47400.0,
  "best_eta_baseline_val_Hit_5m_percent": 0.0,
  "best_eta_baseline_val_Hit_10m_percent": 0.0,
  "best_type_baseline_val": "status_mode_type",
  "best_type_baseline_val_accuracy": 0.47058823529411764,
  "best_type_baseline_val_macro_f1": 0.4631578947368421,
  "best_type_baseline_val_weighted_f1": 0.45944272445820433
}

ETA baseline report:


,split,baseline,MAE_sec,RMSE_sec,MedAE_sec,SMAPE_percent,Hit_2m_percent,Hit_5m_percent,Hit_10m_percent,pred_median_sec,pred_p90_sec,actual_median_sec,actual_p90_sec
0,test_reference,machine_median_eta,32513.333333,41386.079785,30660.0,68.047043,0.0,5.555556,5.555556,60660.0,60660.0,44700.0,92370.0
1,test_reference,machine_status_median_eta,34136.666667,42960.393387,34830.0,70.323603,0.0,0.000000,0.000000,60660.0,69300.0,44700.0,92370.0
2,test_reference,status_median_eta,50665.000000,58326.568989,47670.0,82.198894,0.0,0.000000,0.000000,101250.0,102120.0,44700.0,92370.0
3,test_reference,global_median_eta,48400.000000,54815.235108,57450.0,82.027677,0.0,0.000000,0.000000,87900.0,87900.0,44700.0,92370.0
4,val,machine_status_median_eta,64496.470588,98125.901599,47400.0,96.672616,0.0,0.000000,0.000000,60660.0,69300.0,34800.0,161820.0
5,val,machine_median_eta,63543.529412,96452.673918,47460.0,96.474688,0.0,0.000000,0.000000,60660.0,60660.0,34800.0,161820.0
6,val,status_median_eta,80043.529412,97937.995211,65820.0,109.112945,0.0,0.000000,0.000000,101250.0,102120.0,34800.0,161820.0
7,val,global_median_eta,77964.705882,98280.379228,69000.0,108.144913,0.0,0.000000,0.000000,87900.0,87900.0,34800.0,161820.0



Next-type baseline report:


,split,baseline,accuracy,macro_f1,weighted_f1
0,test_reference,status_mode_type,0.388889,0.371429,0.359788
1,test_reference,global_mode_type,0.444444,0.307692,0.273504
2,test_reference,machine_mode_type,0.444444,0.307692,0.273504
3,test_reference,machine_status_mode_type,0.444444,0.307692,0.273504
4,val,status_mode_type,0.470588,0.463158,0.459443
5,val,global_mode_type,0.470588,0.320000,0.301176
6,val,machine_mode_type,0.470588,0.320000,0.301176
7,val,machine_status_mode_type,0.470588,0.320000,0.301176



Best validation next-type baseline: status_mode_type

Classification report for best validation next-type baseline:


,precision,recall,f1-score,support
alarm_coating,0.500000,0.333333,0.400000,9.000000
alarm_palette,0.454545,0.625000,0.526316,8.000000
accuracy,0.470588,0.470588,0.470588,0.470588
macro avg,0.477273,0.479167,0.463158,17.000000
weighted avg,0.478610,0.470588,0.459443,17.000000



Confusion matrix for best validation next-type baseline:


,pred_alarm_coating,pred_alarm_palette
true_alarm_coating,3,6
true_alarm_palette,3,5


READY_FOR_BATCH_6 = True
Batch 5 complete. Send this output before moving to Batch 6.


In [23]:
# ============================================================
# P4-19 — Baseline evaluation report
# ============================================================

import math
import json
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, eps)) * 100


def hit_rate(y_true, y_pred, tolerance_sec):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return np.mean(np.abs(y_true - y_pred) <= tolerance_sec) * 100


def evaluate_eta_predictions(y_true_raw, y_pred, name, split):
    y_true_raw = np.asarray(y_true_raw, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    rmse = math.sqrt(mean_squared_error(y_true_raw, y_pred))

    return {
        "split": split,
        "baseline": name,
        "MAE_sec": mean_absolute_error(y_true_raw, y_pred),
        "RMSE_sec": rmse,
        "MedAE_sec": median_absolute_error(y_true_raw, y_pred),
        "SMAPE_percent": smape(y_true_raw, y_pred),
        "Hit_2m_percent": hit_rate(y_true_raw, y_pred, 120),
        "Hit_5m_percent": hit_rate(y_true_raw, y_pred, 300),
        "Hit_10m_percent": hit_rate(y_true_raw, y_pred, 600),
        "pred_median_sec": float(np.median(y_pred)),
        "pred_p90_sec": float(np.quantile(y_pred, 0.90)),
        "actual_median_sec": float(np.median(y_true_raw)),
        "actual_p90_sec": float(np.quantile(y_true_raw, 0.90)),
    }


def evaluate_type_predictions(y_true, y_pred, name, split):
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)

    return {
        "split": split,
        "baseline": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


eta_baseline_rows = []

for split_name, df_split, y_true_raw, pred_df in [
    ("val", val_df, y_val_raw, eta_baseline_pred_val),
    ("test_reference", test_df, y_test_raw, eta_baseline_pred_test),
]:
    for col in pred_df.columns:
        eta_baseline_rows.append(
            evaluate_eta_predictions(
                y_true_raw=y_true_raw,
                y_pred=pred_df[col].to_numpy(),
                name=col,
                split=split_name,
            )
        )

eta_baseline_report = pd.DataFrame(eta_baseline_rows)

eta_baseline_report_sorted = eta_baseline_report.sort_values(
    ["split", "MedAE_sec", "Hit_5m_percent"],
    ascending=[True, True, False],
).reset_index(drop=True)

type_baseline_rows = []

for split_name, y_true_type, pred_df in [
    ("val", y_type_val, type_baseline_pred_val),
    ("test_reference", y_type_test, type_baseline_pred_test),
]:
    for col in pred_df.columns:
        type_baseline_rows.append(
            evaluate_type_predictions(
                y_true=y_true_type,
                y_pred=pred_df[col].to_numpy(),
                name=col,
                split=split_name,
            )
        )

type_baseline_report = pd.DataFrame(type_baseline_rows)

type_baseline_report_sorted = type_baseline_report.sort_values(
    ["split", "weighted_f1", "accuracy"],
    ascending=[True, False, False],
).reset_index(drop=True)

# Best validation baselines
best_eta_baseline_name = (
    eta_baseline_report[eta_baseline_report["split"].eq("val")]
    .sort_values(["MedAE_sec", "Hit_5m_percent"], ascending=[True, False])
    .iloc[0]["baseline"]
)

best_type_baseline_name = (
    type_baseline_report[type_baseline_report["split"].eq("val")]
    .sort_values(["weighted_f1", "accuracy"], ascending=[False, False])
    .iloc[0]["baseline"]
)

best_eta_baseline_val = eta_baseline_report[
    (eta_baseline_report["split"].eq("val")) &
    (eta_baseline_report["baseline"].eq(best_eta_baseline_name))
].iloc[0].to_dict()

best_type_baseline_val = type_baseline_report[
    (type_baseline_report["split"].eq("val")) &
    (type_baseline_report["baseline"].eq(best_type_baseline_name))
].iloc[0].to_dict()

# Detailed classification report for best validation type baseline
best_type_val_pred = type_baseline_pred_val[best_type_baseline_name].astype(str).to_numpy()

type_classes_sorted = sorted(set(y_type_val) | set(best_type_val_pred))

best_type_classification_report_dict = classification_report(
    y_type_val,
    best_type_val_pred,
    labels=type_classes_sorted,
    output_dict=True,
    zero_division=0,
)

best_type_classification_report_df = pd.DataFrame(
    best_type_classification_report_dict
).T

best_type_confusion = pd.DataFrame(
    confusion_matrix(y_type_val, best_type_val_pred, labels=type_classes_sorted),
    index=[f"true_{c}" for c in type_classes_sorted],
    columns=[f"pred_{c}" for c in type_classes_sorted],
)

baseline_summary = {
    "best_eta_baseline_val": best_eta_baseline_name,
    "best_eta_baseline_val_MedAE_sec": float(best_eta_baseline_val["MedAE_sec"]),
    "best_eta_baseline_val_Hit_5m_percent": float(best_eta_baseline_val["Hit_5m_percent"]),
    "best_eta_baseline_val_Hit_10m_percent": float(best_eta_baseline_val["Hit_10m_percent"]),
    "best_type_baseline_val": best_type_baseline_name,
    "best_type_baseline_val_accuracy": float(best_type_baseline_val["accuracy"]),
    "best_type_baseline_val_macro_f1": float(best_type_baseline_val["macro_f1"]),
    "best_type_baseline_val_weighted_f1": float(best_type_baseline_val["weighted_f1"]),
}

print("=" * 100)
print("P4-19 — BASELINE EVALUATION REPORT")
print("=" * 100)

# ------------------------------------------------------------
# FANCY RENDER BLOCK FOR STAKEHOLDERS (machine_median_eta)
# ------------------------------------------------------------
MAE_WARN_THRESHOLD = 600.0   # seconds
MEDAE_WARN_THRESHOLD = 450.0 # seconds (adjustable example)

mm_eta_df = eta_baseline_report[eta_baseline_report["baseline"] == "machine_median_eta"]

if not mm_eta_df.empty:
    html_blocks = ["""
    <style>
        .report-box { font-family: 'Segoe UI', Arial, sans-serif; margin: 25px 0; padding: 25px; border-radius: 12px; background-color: #fcfcfd; border: 1px solid #e4e7eb; box-shadow: 0 4px 6px rgba(0,0,0,0.02); }
        .banner { padding: 18px; border-radius: 8px; font-size: 1.25rem; font-weight: bold; margin-bottom: 25px; text-align: center; letter-spacing: 0.5px; }
        .banner-danger { background-color: #fff5f5; color: #c53030; border: 1px solid #feb2b2; }
        .banner-success { background-color: #f0fff4; color: #22543d; border: 1px solid #9ae6b4; }
        .sect-title { font-size: 1.1rem; color: #4a5568; border-bottom: 2px solid #edf2f7; padding-bottom: 6px; margin-top: 20px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;}
        .deck { display: flex; flex-wrap: wrap; gap: 15px; margin-top: 12px; margin-bottom: 15px; }
        .card { flex: 1; min-width: 200px; background: white; padding: 18px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.04); border-top: 4px solid #cbd5e0; }
        .card.danger { border-top-color: #e53e3e; background-color: #fffaf0; }
        .card.success { border-top-color: #38a169; }
        .lbl { font-size: 0.8rem; color: #718096; text-transform: uppercase; font-weight: 600; letter-spacing: 0.5px; }
        .val { font-size: 1.8rem; font-weight: 800; color: #2d3748; margin: 8px 0; }
        .status-txt { font-size: 0.85rem; font-weight: 700; }
        .txt-danger { color: #e53e3e; }
        .txt-success { color: #38a169; }
    </style>
    <div class="report-box">
        <h2 style="margin-top:0; color:#1a202c; border-bottom: 3px solid #3182ce; padding-bottom: 8px;">📋 Executive Dataset Viability Report</h2>
    """]
    
    # Check if either condition fails across any data split
    is_unusable = any((row["MAE_sec"] > MAE_WARN_THRESHOLD or row["MedAE_sec"] > MEDAE_WARN_THRESHOLD) for _, row in mm_eta_df.iterrows())
    
    if is_unusable:
        html_blocks.append(f"""
        <div class="banner banner-danger">
            ⚠️ DATASET UNUSABLE FOR THIS PIPELINE
            <br><span style="font-size:0.95rem; font-weight:normal; display:block; margin-top:5px;">The recorded predictive baseline metrics exceed acceptable tolerances (Max Allowed: MAE &le; {MAE_WARN_THRESHOLD}s, MedAE &le; {MEDAE_WARN_THRESHOLD}s). This indicates high instability/noise in the source data.</span>
        </div>
        """)
    else:
        html_blocks.append("""
        <div class="banner banner-success">
            ✅ DATASET QUALITY ACCEPTABLE
            <br><span style="font-size:0.95rem; font-weight:normal; display:block; margin-top:5px;">All evaluated metrics fall safely within parameters for deployment pipeline operations.</span>
        </div>
        """)
        
    for _, row in mm_eta_df.iterrows():
        split_lbl = "Validation Data Split" if row["split"] == "val" else "Test Reference Data Split"
        mae = row["MAE_sec"]
        medae = row["MedAE_sec"]
        hit_5m = row["Hit_5m_percent"]
        
        mae_state = "danger" if mae > MAE_WARN_THRESHOLD else "success"
        mae_msg = f"❌ Fails (> {MAE_WARN_THRESHOLD}s)" if mae > MAE_WARN_THRESHOLD else "✓ Pass"
        
        medae_state = "danger" if medae > MEDAE_WARN_THRESHOLD else "success"
        medae_msg = f"❌ Fails (> {MEDAE_WARN_THRESHOLD}s)" if medae > MEDAE_WARN_THRESHOLD else "✓ Pass"
        
        html_blocks.append(f"""
        <div class="sect-title">{split_lbl} (machine_median_eta)</div>
        <div class="deck">
            <div class="card {mae_state}">
                <div class="lbl">Mean Absolute Error (MAE)</div>
                <div class="val">{mae:.1f}s</div>
                <div class="status-txt txt-{mae_state}">{mae_msg}</div>
            </div>
            <div class="card {medae_state}">
                <div class="lbl">Median Absolute Error (MedAE)</div>
                <div class="val">{medae:.1f}s</div>
                <div class="status-txt txt-{medae_state}">{medae_msg}</div>
            </div>
            <div class="card success">
                <div class="lbl">Prediction Hit Rate (Within 5m)</div>
                <div class="val">{hit_5m:.1f}%</div>
                <div class="status-txt txt-success">Performance Tracking (should be >75%)</div>
            </div>
        </div>
        """)
        
    html_blocks.append("</div>")
    display(HTML("".join(html_blocks)))
else:
    print("\n[Warning] 'machine_median_eta' baseline not detected in generated dataframe results.")


print("\nBaseline summary:")
print(json.dumps(baseline_summary, indent=2, ensure_ascii=False))

print("\nETA baseline report:")
display(eta_baseline_report_sorted)

print("\nNext-type baseline report:")
display(type_baseline_report_sorted)

print(f"\nBest validation next-type baseline: {best_type_baseline_name}")
print("\nClassification report for best validation next-type baseline:")
display(best_type_classification_report_df)

print("\nConfusion matrix for best validation next-type baseline:")
display(best_type_confusion)

print("=" * 100)
print("READY_FOR_BATCH_6 = True")
print("Batch 5 complete. Send this output before moving to Batch 6.")
print("=" * 100)

P4-19 — BASELINE EVALUATION REPORT



Baseline summary:
{
  "best_eta_baseline_val": "machine_status_median_eta",
  "best_eta_baseline_val_MedAE_sec": 47400.0,
  "best_eta_baseline_val_Hit_5m_percent": 0.0,
  "best_eta_baseline_val_Hit_10m_percent": 0.0,
  "best_type_baseline_val": "status_mode_type",
  "best_type_baseline_val_accuracy": 0.47058823529411764,
  "best_type_baseline_val_macro_f1": 0.4631578947368421,
  "best_type_baseline_val_weighted_f1": 0.45944272445820433
}

ETA baseline report:


,split,baseline,MAE_sec,RMSE_sec,MedAE_sec,SMAPE_percent,Hit_2m_percent,Hit_5m_percent,Hit_10m_percent,pred_median_sec,pred_p90_sec,actual_median_sec,actual_p90_sec
0,test_reference,machine_median_eta,32513.333333,41386.079785,30660.0,68.047043,0.0,5.555556,5.555556,60660.0,60660.0,44700.0,92370.0
1,test_reference,machine_status_median_eta,34136.666667,42960.393387,34830.0,70.323603,0.0,0.000000,0.000000,60660.0,69300.0,44700.0,92370.0
2,test_reference,status_median_eta,50665.000000,58326.568989,47670.0,82.198894,0.0,0.000000,0.000000,101250.0,102120.0,44700.0,92370.0
3,test_reference,global_median_eta,48400.000000,54815.235108,57450.0,82.027677,0.0,0.000000,0.000000,87900.0,87900.0,44700.0,92370.0
4,val,machine_status_median_eta,64496.470588,98125.901599,47400.0,96.672616,0.0,0.000000,0.000000,60660.0,69300.0,34800.0,161820.0
5,val,machine_median_eta,63543.529412,96452.673918,47460.0,96.474688,0.0,0.000000,0.000000,60660.0,60660.0,34800.0,161820.0
6,val,status_median_eta,80043.529412,97937.995211,65820.0,109.112945,0.0,0.000000,0.000000,101250.0,102120.0,34800.0,161820.0
7,val,global_median_eta,77964.705882,98280.379228,69000.0,108.144913,0.0,0.000000,0.000000,87900.0,87900.0,34800.0,161820.0



Next-type baseline report:


,split,baseline,accuracy,macro_f1,weighted_f1
0,test_reference,status_mode_type,0.388889,0.371429,0.359788
1,test_reference,global_mode_type,0.444444,0.307692,0.273504
2,test_reference,machine_mode_type,0.444444,0.307692,0.273504
3,test_reference,machine_status_mode_type,0.444444,0.307692,0.273504
4,val,status_mode_type,0.470588,0.463158,0.459443
5,val,global_mode_type,0.470588,0.320000,0.301176
6,val,machine_mode_type,0.470588,0.320000,0.301176
7,val,machine_status_mode_type,0.470588,0.320000,0.301176



Best validation next-type baseline: status_mode_type

Classification report for best validation next-type baseline:


,precision,recall,f1-score,support
alarm_coating,0.500000,0.333333,0.400000,9.000000
alarm_palette,0.454545,0.625000,0.526316,8.000000
accuracy,0.470588,0.470588,0.470588,0.470588
macro avg,0.477273,0.479167,0.463158,17.000000
weighted avg,0.478610,0.470588,0.459443,17.000000



Confusion matrix for best validation next-type baseline:


,pred_alarm_coating,pred_alarm_palette
true_alarm_coating,3,6
true_alarm_palette,3,5


READY_FOR_BATCH_6 = True
Batch 5 complete. Send this output before moving to Batch 6.
